In [1]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from CBFV import composition
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import re
from ast import literal_eval
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from ax.service.ax_client import AxClient, ObjectiveProperties
from ax.core.arm import Arm

In [ ]:
#function used to parse Ax logs and reconstruct an AxClient with the same trials and results, useful for continuing optimization after a long run or crash
def assemble_ax_from_log(ax_client, log_path, save_path):

    OBJECTIVE_NAME = "avg_rmse_nonzero"  # the metric you want to extract for optimization
    MINIMIZE = True  # RMSE -> minimize

    # --------- 2) Parse log ----------
    gen_re = re.compile(
        r"Generated new trial\s+(?P<trial>\d+)\s+with parameters\s+(?P<params>\{.*?\})\s+using model",
        re.DOTALL
    )

    # Handles: {'avg_rmse_nonzero': (np.float32(0.377...), np.float64(0.032...))}
    # Also handles: {'avg_rmse_nonzero': (0.377..., 0.032...)}
    comp_re = re.compile(
        r"Completed trial\s+(?P<trial>\d+)\s+with data:\s+(?P<data>\{.*?\})\.",
        re.DOTALL
    )

    def _strip_np_wrappers(s: str) -> str:
        # Turn np.float32(0.1) -> 0.1, np.float64(0.2) -> 0.2, etc.
        s = re.sub(r"np\.float(?:16|32|64)\(([^)]+)\)", r"\1", s)
        s = re.sub(r"np\.int(?:8|16|32|64)\(([^)]+)\)", r"\1", s)
        return s

    with open(log_path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    generated = {}
    for m in gen_re.finditer(text):
        t = int(m["trial"])
        params_str = m["params"]
        params = literal_eval(params_str)  # safe for dict literals like yours
        generated[t] = params

    completed = {}
    for m in comp_re.finditer(text):
        t = int(m["trial"])
        data_str = _strip_np_wrappers(m["data"])
        data = literal_eval(data_str)
        # Expect metric -> (mean, sem) OR metric -> mean
        val = data[OBJECTIVE_NAME]
        if isinstance(val, tuple):
            mean, sem = float(val[0]), float(val[1])
        else:
            mean, sem = float(val), None
        completed[t] = (mean, sem)

    # Align trials that have both params and completion
    trial_ids = sorted(set(generated.keys()) & set(completed.keys()))
    missing_params = sorted(set(completed.keys()) - set(generated.keys()))
    missing_results = sorted(set(generated.keys()) - set(completed.keys()))

    print(f"Recovered {len(trial_ids)} completed trials with params.")
    if missing_params:
        print(f"Warning: {len(missing_params)} completed trials missing params: {missing_params[:10]} ...")
    if missing_results:
        print(f"Note: {len(missing_results)} generated trials missing results (not completed): {missing_results[:10]} ...")

        exp = ax_client.experiment

    for t in trial_ids:
        params = generated[t]
        mean, sem = completed[t]

        trial = exp.new_trial()
        trial.add_arm(Arm(parameters=params))

        # IMPORTANT for your Ax version:
        trial.mark_running(no_runner_required=True)

        if sem is None:
            ax_client.complete_trial(
                trial_index=trial.index,
                raw_data={OBJECTIVE_NAME: mean},
            )
        else:
            ax_client.complete_trial(
                trial_index=trial.index,
                raw_data={OBJECTIVE_NAME: (mean, sem)},
            )


In [2]:
class FlexibleNN(nn.Module):
    """
    A flexible neural network with configurable architecture for hyperparameter optimization.
    
    Parameters:
    -----------
    input_dim : int
        Number of input features (CBFV features)
    output_dim : int
        Number of output targets (phase fractions at selected temperatures)
    hidden_layers : list of int
        List of hidden layer sizes, e.g., [256, 128, 64]
    dropout_rate : float
        Dropout probability (0.0 to 1.0)
    dropout_type : str
        Type of dropout: 'standard', 'alpha' (for SELU), or 'none'
    activation : str
        Activation function: 'relu', 'leaky_relu', 'elu', 'selu', 'gelu', 'tanh'
    use_batch_norm : bool
        Whether to use batch normalization
    use_layer_norm : bool
        Whether to use layer normalization (alternative to batch norm)
    weight_decay : float
        L2 regularization strength (applied in optimizer, stored here for reference)
    nf_indices : list of int or None
        Indices of output columns that are phase fractions (NF). 
        These will have softmax applied so they sum to 1.
        If None, no softmax constraint is applied.
    output_columns : list of str or None
        Column names for outputs. If provided, NF columns are auto-detected
        by checking for 'NF_' prefix. Overrides nf_indices if both provided.
    """
    
    def __init__(
        self,
        input_dim,
        output_dim,
        hidden_layers=[256, 128, 64],
        dropout_rate=0.2,
        dropout_type='standard',
        activation='relu',
        use_batch_norm=False,
        use_layer_norm=False,
        weight_decay=0.0,
        nf_indices=None,
        output_columns=None
    ):
        super(FlexibleNN, self).__init__()
        
        self.weight_decay = weight_decay  # Store for optimizer configuration
        self.output_dim = output_dim
        
        # Determine NF indices (phase fractions that should sum to 1)
        if output_columns is not None:
            # Auto-detect NF columns from column names
            self.nf_indices = [i for i, col in enumerate(output_columns) if col.startswith('NF_')]
            self.df_indices = [i for i, col in enumerate(output_columns) if col.startswith('DF_')]
        elif nf_indices is not None:
            self.nf_indices = nf_indices
            self.df_indices = [i for i in range(output_dim) if i not in nf_indices]
        else:
            self.nf_indices = []
            self.df_indices = list(range(output_dim))
        
        # Build activation function
        activation_fn = self._get_activation(activation)
        
        # Build dropout layer
        dropout_layer = self._get_dropout(dropout_type, dropout_rate)
        
        # Build the network layers
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_layers:
            # Linear layer
            layers.append(nn.Linear(prev_dim, hidden_dim))
            
            # Normalization (batch norm or layer norm, not both)
            if use_batch_norm:
                layers.append(nn.BatchNorm1d(hidden_dim))
            elif use_layer_norm:
                layers.append(nn.LayerNorm(hidden_dim))
            
            # Activation
            layers.append(activation_fn())
            
            # Dropout
            if dropout_layer is not None:
                layers.append(dropout_layer(dropout_rate))
            
            prev_dim = hidden_dim
        
        # Output layer (no activation, dropout, or normalization)
        layers.append(nn.Linear(prev_dim, output_dim))
        
        self.network = nn.Sequential(*layers)
    
    def _get_activation(self, activation):
        activations = {
            'relu': nn.ReLU,
            'leaky_relu': lambda: nn.LeakyReLU(0.1),
            'elu': nn.ELU,
            'selu': nn.SELU,
            'gelu': nn.GELU,
            'tanh': nn.Tanh,
            'sigmoid': nn.Sigmoid
        }
        if activation not in activations:
            raise ValueError(f"Unknown activation: {activation}. Choose from {list(activations.keys())}")
        return activations[activation]
    
    def _get_dropout(self, dropout_type, dropout_rate):
        if dropout_type == 'none' or dropout_rate == 0:
            return None
        elif dropout_type == 'standard':
            return nn.Dropout
        elif dropout_type == 'alpha':
            return nn.AlphaDropout  # For use with SELU activation
        else:
            raise ValueError(f"Unknown dropout type: {dropout_type}. Choose from ['standard', 'alpha', 'none']")
    
    def forward(self, x):
        raw_output = self.network(x)
        
        # If no NF indices, return raw output
        if len(self.nf_indices) == 0:
            return raw_output
        
        # Apply softmax to NF (phase fraction) outputs so they sum to 1
        output = raw_output.clone()
        
        # Extract NF outputs and apply softmax
        nf_outputs = raw_output[:, self.nf_indices]
        nf_normalized = torch.softmax(nf_outputs, dim=1)
        
        # Put normalized NF values back
        output[:, self.nf_indices] = nf_normalized
        
        return output
    
    def get_optimizer(self, optimizer_type='adam', lr=1e-3):
        """
        Get an optimizer with the configured weight decay (L2 regularization).
        """
        optimizers = {
            'adam': lambda: torch.optim.Adam(self.parameters(), lr=lr, weight_decay=self.weight_decay),
            'adamw': lambda: torch.optim.AdamW(self.parameters(), lr=lr, weight_decay=self.weight_decay),
            'sgd': lambda: torch.optim.SGD(self.parameters(), lr=lr, weight_decay=self.weight_decay, momentum=0.9),
            'rmsprop': lambda: torch.optim.RMSprop(self.parameters(), lr=lr, weight_decay=self.weight_decay)
        }
        if optimizer_type not in optimizers:
            raise ValueError(f"Unknown optimizer: {optimizer_type}. Choose from {list(optimizers.keys())}")
        return optimizers[optimizer_type]()


def train_epoch(model, train_loader, optimizer, criterion, device):
    """
    Train the model for one epoch.
    
    Returns:
    --------
    float : Average training loss for the epoch
    """
    model.train()
    total_loss = 0.0
    num_batches = 0
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
    
    return total_loss / num_batches


def evaluate_epoch(model, val_loader, criterion, device):
    """
    Evaluate the model on validation data.
    
    Returns:
    --------
    tuple : (average loss, predictions, targets)
    """
    model.eval()
    total_loss = 0.0
    total_samples = 0
    all_predictions = []
    all_targets = []
    
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            batch_size = X_batch.size(0)
            
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            
            # Weight loss by batch size for correct averaging
            total_loss += loss.item() * batch_size
            total_samples += batch_size
            
            all_predictions.append(predictions.cpu())
            all_targets.append(y_batch.cpu())
    
    # Weighted average loss (accounts for different batch sizes)
    avg_loss = total_loss / total_samples
    all_predictions = torch.cat(all_predictions, dim=0)
    all_targets = torch.cat(all_targets, dim=0)
    
    return avg_loss, all_predictions, all_targets


def train_model(
    model,
    X_train, y_train,
    X_val, y_val,
    epochs=100,
    batch_size=32,
    optimizer_type='adam',
    lr=1e-3,
    criterion=None,
    early_stopping_patience=None,
    verbose=True,
    device=None
):
    """
    Train the model for a specified number of epochs.
    
    Parameters:
    -----------
    model : FlexibleNN
        The neural network model
    X_train, y_train : array-like
        Training data
    X_val, y_val : array-like
        Validation data
    epochs : int
        Number of training epochs
    batch_size : int
        Batch size for training
    optimizer_type : str
        Type of optimizer ('adam', 'adamw', 'sgd', 'rmsprop')
    lr : float
        Learning rate
    criterion : nn.Module
        Loss function (default: MSELoss)
    early_stopping_patience : int or None
        Stop training if val loss doesn't improve for this many epochs
    verbose : bool
        Whether to print progress
    device : str or None
        Device to train on ('cuda', 'mps', 'cpu', or None for auto-detect)
    
    Returns:
    --------
    dict : Training history with train_losses, val_losses, best_epoch
    """
    # Auto-detect device
    if device is None:
        if torch.cuda.is_available():
            device = 'cuda'
        elif torch.backends.mps.is_available():
            device = 'mps'
        else:
            device = 'cpu'
    
    device = torch.device(device)
    model = model.to(device)
    
    # Default criterion
    if criterion is None:
        criterion = nn.MSELoss()
    
    # Convert data to tensors
    X_train_t = torch.FloatTensor(X_train.values if hasattr(X_train, 'values') else X_train)
    y_train_t = torch.FloatTensor(y_train.values if hasattr(y_train, 'values') else y_train)
    X_val_t = torch.FloatTensor(X_val.values if hasattr(X_val, 'values') else X_val)
    y_val_t = torch.FloatTensor(y_val.values if hasattr(y_val, 'values') else y_val)
    
    # Create data loaders
    train_dataset = TensorDataset(X_train_t, y_train_t)
    val_dataset = TensorDataset(X_val_t, y_val_t)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    # Get optimizer
    optimizer = model.get_optimizer(optimizer_type=optimizer_type, lr=lr)
    
    # Training history
    history = {
        'train_losses': [],
        'val_losses': [],
        'best_epoch': 0,
        'best_val_loss': float('inf')
    }
    
    # Early stopping
    patience_counter = 0
    best_model_state = None
    
    for epoch in range(epochs):
        # Train
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        
        # Evaluate
        val_loss, _, _ = evaluate_epoch(model, val_loader, criterion, device)
        
        history['train_losses'].append(train_loss)
        history['val_losses'].append(val_loss)
        
        # Track best model
        if val_loss < history['best_val_loss']:
            history['best_val_loss'] = val_loss
            history['best_epoch'] = epoch
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        
        # Verbose output
        if verbose and (epoch % 10 == 0 or epoch == epochs - 1):
            print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.6f} - Val Loss: {val_loss:.6f}")
        
        # Early stopping
        if early_stopping_patience and patience_counter >= early_stopping_patience:
            if verbose:
                print(f"Early stopping at epoch {epoch+1}. Best epoch: {history['best_epoch']+1}")
            break
    
    # Restore best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return history

In [3]:
class PhaseFractionNN(nn.Module):
    """
    Neural network for predicting phase fractions (NF) where outputs must sum to 1.
    Applies softmax to all outputs to ensure they represent valid phase fractions.
    
    Parameters:
    -----------
    input_dim : int
        Number of input features (CBFV features + temperature)
    output_dim : int
        Number of phases to predict
    hidden_layers : list of int
        List of hidden layer sizes, e.g., [256, 512, 1024]
    dropout_rate : float
        Dropout probability (0.0 to 1.0)
    dropout_type : str
        Type of dropout: 'standard', 'alpha' (for SELU), or 'none'
    activation : str
        Activation function: 'relu', 'leaky_relu', 'elu', 'selu', 'gelu', 'tanh'
    use_batch_norm : bool
        Whether to use batch normalization
    use_layer_norm : bool
        Whether to use layer normalization (alternative to batch norm)
    weight_decay : float
        L2 regularization strength (applied in optimizer, stored here for reference)
    """
    
    def __init__(
        self,
        input_dim,
        output_dim,
        hidden_layers=[256, 512, 1024],
        dropout_rate=0.2,
        dropout_type='standard',
        activation='relu',
        use_batch_norm=False,
        use_layer_norm=False,
        weight_decay=0.0,
    ):
        super(PhaseFractionNN, self).__init__()
        
        self.weight_decay = weight_decay
        self.output_dim = output_dim
        
        # Build activation function
        activation_fn = self._get_activation(activation)
        
        # Build dropout layer
        dropout_layer = self._get_dropout(dropout_type, dropout_rate)
        
        # Build the network layers
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_layers:
            # Linear layer
            layers.append(nn.Linear(prev_dim, hidden_dim))
            
            # Normalization (batch norm or layer norm, not both)
            if use_batch_norm:
                layers.append(nn.BatchNorm1d(hidden_dim))
            elif use_layer_norm:
                layers.append(nn.LayerNorm(hidden_dim))
            
            # Activation
            layers.append(activation_fn())
            
            # Dropout
            if dropout_layer is not None:
                layers.append(dropout_layer(dropout_rate))
            
            prev_dim = hidden_dim
        
        # Output layer (no activation here - softmax applied in forward)
        layers.append(nn.Linear(prev_dim, output_dim))
        
        self.network = nn.Sequential(*layers)
    
    def _get_activation(self, activation):
        activations = {
            'relu': nn.ReLU,
            'leaky_relu': lambda: nn.LeakyReLU(0.1),
            'elu': nn.ELU,
            'selu': nn.SELU,
            'gelu': nn.GELU,
            'tanh': nn.Tanh,
            'sigmoid': nn.Sigmoid
        }
        if activation not in activations:
            raise ValueError(f"Unknown activation: {activation}. Choose from {list(activations.keys())}")
        return activations[activation]
    
    def _get_dropout(self, dropout_type, dropout_rate):
        if dropout_type == 'none' or dropout_rate == 0:
            return None
        elif dropout_type == 'standard':
            return nn.Dropout
        elif dropout_type == 'alpha':
            return nn.AlphaDropout  # For use with SELU activation
        else:
            raise ValueError(f"Unknown dropout type: {dropout_type}. Choose from ['standard', 'alpha', 'none']")
    
    def forward(self, x):
        """
        Forward pass with softmax applied to ensure outputs sum to 1.
        """
        logits = self.network(x)
        # Apply softmax so phase fractions sum to 1
        phase_fractions = torch.softmax(logits, dim=1)
        return phase_fractions
    
    def get_optimizer(self, optimizer_type='adam', lr=1e-3):
        """
        Get an optimizer with the configured weight decay (L2 regularization).
        """
        optimizers = {
            'adam': lambda: torch.optim.Adam(self.parameters(), lr=lr, weight_decay=self.weight_decay),
            'adamw': lambda: torch.optim.AdamW(self.parameters(), lr=lr, weight_decay=self.weight_decay),
            'sgd': lambda: torch.optim.SGD(self.parameters(), lr=lr, weight_decay=self.weight_decay, momentum=0.9),
            'rmsprop': lambda: torch.optim.RMSprop(self.parameters(), lr=lr, weight_decay=self.weight_decay)
        }
        if optimizer_type not in optimizers:
            raise ValueError(f"Unknown optimizer: {optimizer_type}. Choose from {list(optimizers.keys())}")
        return optimizers[optimizer_type]()





# Custom loss for phase fractions - can use KL divergence or cross-entropy style loss
class PhaseFractionLoss(nn.Module):
    """
    Loss function for phase fraction prediction.
    Combines MSE with optional KL divergence to encourage proper probability distributions.
    
    Parameters:
    -----------
    mse_weight : float
        Weight for MSE component (default: 1.0)
    kl_weight : float
        Weight for KL divergence component (default: 0.0)
        Set > 0 to encourage outputs to match target distribution shape
    """
    def __init__(self, mse_weight=1.0, kl_weight=0.0):
        super().__init__()
        self.mse_weight = mse_weight
        self.kl_weight = kl_weight
        self.mse = nn.MSELoss()
        self.kl = nn.KLDivLoss(reduction='batchmean')
    
    def forward(self, predictions, targets):
        # MSE loss
        mse_loss = self.mse(predictions, targets)
        
        if self.kl_weight > 0:
            # KL divergence (predictions should already be softmax output)
            # Add small epsilon to avoid log(0)
            eps = 1e-8
            log_preds = torch.log(predictions + eps)
            # Normalize targets to sum to 1 (they should already, but just in case)
            targets_norm = targets / (targets.sum(dim=1, keepdim=True) + eps)
            kl_loss = self.kl(log_preds, targets_norm)
            return self.mse_weight * mse_loss + self.kl_weight * kl_loss
        
        return mse_loss


class DrivingForceNN(nn.Module):
    """
    Neural network for predicting driving forces (DF).
    Outputs are unconstrained real values (no softmax).
    
    Parameters:
    -----------
    input_dim : int
        Number of input features (CBFV features + temperature)
    output_dim : int
        Number of driving force outputs to predict
    hidden_layers : list of int
        List of hidden layer sizes, e.g., [256, 512, 1024]
    dropout_rate : float
        Dropout probability (0.0 to 1.0)
    dropout_type : str
        Type of dropout: 'standard', 'alpha' (for SELU), or 'none'
    activation : str
        Activation function: 'relu', 'leaky_relu', 'elu', 'selu', 'gelu', 'tanh'
    use_batch_norm : bool
        Whether to use batch normalization
    use_layer_norm : bool
        Whether to use layer normalization (alternative to batch norm)
    weight_decay : float
        L2 regularization strength (applied in optimizer, stored here for reference)
    """
    
    def __init__(
        self,
        input_dim,
        output_dim,
        hidden_layers=[256, 512, 1024],
        dropout_rate=0.2,
        dropout_type='standard',
        activation='relu',
        use_batch_norm=False,
        use_layer_norm=False,
        weight_decay=0.0,
    ):
        super(DrivingForceNN, self).__init__()
        
        self.weight_decay = weight_decay
        self.output_dim = output_dim
        
        # Build activation function
        activation_fn = self._get_activation(activation)
        
        # Build dropout layer
        dropout_layer = self._get_dropout(dropout_type, dropout_rate)
        
        # Build the network layers
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_layers:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            
            if use_batch_norm:
                layers.append(nn.BatchNorm1d(hidden_dim))
            elif use_layer_norm:
                layers.append(nn.LayerNorm(hidden_dim))
            
            layers.append(activation_fn())
            
            if dropout_layer is not None:
                layers.append(dropout_layer(dropout_rate))
            
            prev_dim = hidden_dim
        
        # Output layer - no activation (driving forces are unconstrained real values)
        layers.append(nn.Linear(prev_dim, output_dim))
        
        self.network = nn.Sequential(*layers)
    
    def _get_activation(self, activation):
        activations = {
            'relu': nn.ReLU,
            'leaky_relu': lambda: nn.LeakyReLU(0.1),
            'elu': nn.ELU,
            'selu': nn.SELU,
            'gelu': nn.GELU,
            'tanh': nn.Tanh,
            'sigmoid': nn.Sigmoid
        }
        if activation not in activations:
            raise ValueError(f"Unknown activation: {activation}. Choose from {list(activations.keys())}")
        return activations[activation]
    
    def _get_dropout(self, dropout_type, dropout_rate):
        if dropout_type == 'none' or dropout_rate == 0:
            return None
        elif dropout_type == 'standard':
            return nn.Dropout
        elif dropout_type == 'alpha':
            return nn.AlphaDropout
        else:
            raise ValueError(f"Unknown dropout type: {dropout_type}. Choose from ['standard', 'alpha', 'none']")
    
    def forward(self, x):
        """
        Forward pass - raw output, no constraints applied.
        Driving forces can be any real value.
        """
        return self.network(x)
    
    def get_optimizer(self, optimizer_type='adam', lr=1e-3):
        """
        Get an optimizer with the configured weight decay (L2 regularization).
        """
        optimizers = {
            'adam': lambda: torch.optim.Adam(self.parameters(), lr=lr, weight_decay=self.weight_decay),
            'adamw': lambda: torch.optim.AdamW(self.parameters(), lr=lr, weight_decay=self.weight_decay),
            'sgd': lambda: torch.optim.SGD(self.parameters(), lr=lr, weight_decay=self.weight_decay, momentum=0.9),
            'rmsprop': lambda: torch.optim.RMSprop(self.parameters(), lr=lr, weight_decay=self.weight_decay)
        }
        if optimizer_type not in optimizers:
            raise ValueError(f"Unknown optimizer: {optimizer_type}. Choose from {list(optimizers.keys())}")
        return optimizers[optimizer_type]()

In [4]:
def fourier_features(T, n_freqs=6):
    """
    Generate Fourier features for temperature values.
    
    Args:
        T: 1D tensor of temperature values (shape: [N])
        n_freqs: number of frequency components (produces 2*n_freqs features)
    
    Returns:
        2D tensor of shape [N, 2*n_freqs] where each row is the Fourier encoding
    """
    # Ensure T is 2D with shape [N, 1] for broadcasting
    if T.dim() == 1:
        T = T.unsqueeze(1)
    
    feats = []
    for k in range(n_freqs):
        freq = 2**k
        feats.append(torch.sin(2 * torch.pi * freq * T))
        feats.append(torch.cos(2 * torch.pi * freq * T))
    
    # Stack along dim=1 to get shape [N, 2*n_freqs]
    return torch.cat(feats, dim=1)

In [ ]:
df_reshaped = pd.read_csv(r'Data/F(T)_Data/calphad_alloys_train_opt_reshaped.csv')
df_reshaped.head()

,alloy_string,temperature,DF_AG2CA,DF_AG3BE8,DF_AG3CA5,DF_AG3MG,DF_AG7CA2,DF_AG9CA2,DF_AGCA,DF_AGCA3,...,NF_YSI2_H,NF_YSI2_R,NF_ZINCBLENDE_B3,NF_ZR2SI,NF_ZR3SI,NF_ZR3SI2,NF_ZR5SI3,NF_ZR5SI4,NF_ZRSI,NF_ZRSI2
0,B22.00Co4.00Fe68.00Y6.00,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,B22.00Co4.00Fe68.00Y6.00,50,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,B22.00Co4.00Fe68.00Y6.00,100,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,B22.00Co4.00Fe68.00Y6.00,150,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,B22.00Co4.00Fe68.00Y6.00,200,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
# Extract alloy strings and create formula dataframe
formula_df = pd.DataFrame({'formula': df_reshaped['alloy_string']})
print(f"Formula dataframe shape: {formula_df.shape}")
formula_df.head()

Formula dataframe shape: (44982, 1)


,formula
0,B22.00Co4.00Fe68.00Y6.00
1,B22.00Co4.00Fe68.00Y6.00
2,B22.00Co4.00Fe68.00Y6.00
3,B22.00Co4.00Fe68.00Y6.00
4,B22.00Co4.00Fe68.00Y6.00


In [7]:
# Generate Composition-Based Feature Vectors using CBFV
# CBFV expects 'formula' column and optionally a 'target' column
# Add a dummy target for featurization (we'll drop it after)
formula_df['target'] = 0

# Generate CBFVs using the magpie element property database
X, y, formulae, skipped = composition.generate_features(formula_df, elem_prop='magpie')
print(f"CBFV features shape: {X.shape}")
print(f"Number of skipped formulas: {len(skipped)}")
X.head()

Processing Input Data: 100%|██████████| 44982/44982 [00:01<00:00, 38269.41it/s]


	Featurizing Compositions...


Assigning Features...: 100%|██████████| 44982/44982 [00:01<00:00, 31641.54it/s]


	Creating Pandas Objects...
CBFV features shape: (44982, 132)
Number of skipped formulas: 0


,avg_Number,avg_MendeleevNumber,avg_AtomicWeight,avg_MeltingT,avg_Column,avg_Row,avg_CovalentRadius,avg_Electronegativity,avg_NsValence,avg_NpValence,...,mode_NValence,mode_NsUnfilled,mode_NpUnfilled,mode_NdUnfilled,mode_NfUnfilled,mode_NUnfilled,mode_GSvolume_pa,mode_GSbandgap,mode_GSmagmom,mode_SpaceGroupNumber
0,22.2,56.28,48.044699,1926.7,8.84,3.62,124.68,1.8416,2.0,0.22,...,8.0,0.0,0.0,4.0,0.0,4.0,10.73,0.0,2.110663,229.0
1,22.2,56.28,48.044699,1926.7,8.84,3.62,124.68,1.8416,2.0,0.22,...,8.0,0.0,0.0,4.0,0.0,4.0,10.73,0.0,2.110663,229.0
2,22.2,56.28,48.044699,1926.7,8.84,3.62,124.68,1.8416,2.0,0.22,...,8.0,0.0,0.0,4.0,0.0,4.0,10.73,0.0,2.110663,229.0
3,22.2,56.28,48.044699,1926.7,8.84,3.62,124.68,1.8416,2.0,0.22,...,8.0,0.0,0.0,4.0,0.0,4.0,10.73,0.0,2.110663,229.0
4,22.2,56.28,48.044699,1926.7,8.84,3.62,124.68,1.8416,2.0,0.22,...,8.0,0.0,0.0,4.0,0.0,4.0,10.73,0.0,2.110663,229.0


In [8]:
fourier_Temp_features = fourier_features(torch.tensor(df_reshaped['temperature'].values, dtype=torch.float32))
fourier_Temp_features_df = pd.DataFrame(
    fourier_Temp_features.numpy(),
    columns=[f'fourier_{i}' for i in range(fourier_Temp_features.shape[1])],
    index=df_reshaped.index
)

In [9]:
X_combined = pd.concat([X, fourier_Temp_features_df], axis=1)
print(f"X shape: {X_combined.shape}")
X_combined.head()

X shape: (44982, 144)


,avg_Number,avg_MendeleevNumber,avg_AtomicWeight,avg_MeltingT,avg_Column,avg_Row,avg_CovalentRadius,avg_Electronegativity,avg_NsValence,avg_NpValence,...,fourier_2,fourier_3,fourier_4,fourier_5,fourier_6,fourier_7,fourier_8,fourier_9,fourier_10,fourier_11
0,22.2,56.28,48.044699,1926.7,8.84,3.62,124.68,1.8416,2.0,0.22,...,0.000000,1.0,0.000000,1.0,0.000000,1.0,0.000000,1.0,0.000000,1.000000
1,22.2,56.28,48.044699,1926.7,8.84,3.62,124.68,1.8416,2.0,0.22,...,0.000012,1.0,0.000024,1.0,0.000047,1.0,0.000094,1.0,0.000188,1.000000
2,22.2,56.28,48.044699,1926.7,8.84,3.62,124.68,1.8416,2.0,0.22,...,0.000024,1.0,0.000047,1.0,0.000094,1.0,0.000188,1.0,0.000376,1.000000
3,22.2,56.28,48.044699,1926.7,8.84,3.62,124.68,1.8416,2.0,0.22,...,0.000096,1.0,0.000193,1.0,0.000385,1.0,0.000771,1.0,0.001541,0.999999
4,22.2,56.28,48.044699,1926.7,8.84,3.62,124.68,1.8416,2.0,0.22,...,0.000047,1.0,0.000094,1.0,0.000188,1.0,0.000376,1.0,0.000753,1.000000


In [10]:
# Create 5 CV groups by clustering similar alloys together
# Standardize features for clustering
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_combined)

# Use KMeans to group similar alloys into 5 clusters
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
cv_groups = kmeans.fit_predict(X_scaled)

#local CV, umap,

# Add group assignments to a dataframe for reference
cv_group_df = pd.DataFrame({
    'formula': formulae,
    'cv_group': cv_groups
})

print("CV Group distribution:")
print(cv_group_df['cv_group'].value_counts().sort_index())
print(f"\nTotal samples: {len(cv_groups)}")
cv_group_df.head(10)

CV Group distribution:
cv_group
0     7599
1    16014
2     6375
3    10200
4     4794
Name: count, dtype: int64

Total samples: 44982


,formula,cv_group
0,B22.00Co4.00Fe68.00Y6.00,3
1,B22.00Co4.00Fe68.00Y6.00,3
2,B22.00Co4.00Fe68.00Y6.00,3
3,B22.00Co4.00Fe68.00Y6.00,3
4,B22.00Co4.00Fe68.00Y6.00,3
5,B22.00Co4.00Fe68.00Y6.00,3
6,B22.00Co4.00Fe68.00Y6.00,3
7,B22.00Co4.00Fe68.00Y6.00,3
8,B22.00Co4.00Fe68.00Y6.00,3
9,B22.00Co4.00Fe68.00Y6.00,3


In [11]:
# Create separate y dataframes for NF (phase fractions) and DF (driving forces)
all_cols = df_reshaped.drop(columns=['alloy_string', 'temperature']).columns.tolist()

# Split into NF and DF columns
nf_cols = [col for col in all_cols if col.startswith('NF_')]
df_cols = [col for col in all_cols if col.startswith('DF_')]

y_nf = df_reshaped[nf_cols]
y_df = df_reshaped[df_cols]

print(f"Phase Fractions (NF) shape: {y_nf.shape}")
print(f"Driving Forces (DF) shape: {y_df.shape}")
print(f"\nNF columns: {len(nf_cols)}, DF columns: {len(df_cols)}")
y_nf.head()

Phase Fractions (NF) shape: (44982, 804)
Driving Forces (DF) shape: (44982, 804)

NF columns: 804, DF columns: 804


,NF_AG2CA,NF_AG3BE8,NF_AG3CA5,NF_AG3MG,NF_AG7CA2,NF_AG9CA2,NF_AGCA,NF_AGCA3,NF_AGCD_ETA,NF_AGIN2,...,NF_YSI2_H,NF_YSI2_R,NF_ZINCBLENDE_B3,NF_ZR2SI,NF_ZR3SI,NF_ZR3SI2,NF_ZR5SI3,NF_ZR5SI4,NF_ZRSI,NF_ZRSI2
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
def evaluate_parameters_NFnn(parameters, batch_size=32, epochs=1000, verbose=True):
    
    # Extract hyperparameters from parameters dict with defaults
    batch_size = parameters.get('batch_size', batch_size)
    hidden_layers = parameters.get('hidden_layers', [256, 128, 64])
    dropout_rate = parameters.get('dropout_rate', 0.2)
    dropout_type = parameters.get('dropout_type', 'standard')
    activation = parameters.get('activation', 'relu')
    use_batch_norm = parameters.get('use_batch_norm', False)
    use_layer_norm = parameters.get('use_layer_norm', False)
    weight_decay = parameters.get('weight_decay', 0.0)
    optimizer_type = parameters.get('optimizer_type', 'adam')
    lr = parameters.get('lr', 1e-3)
    early_stopping_patience = parameters.get('early_stopping_patience', 10)

    # Auto-detect device
    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device('cpu')
    print(f"Using device: {device}")

    #copy the x and y data
    y_data = y_nf.copy()
    x_data = X_combined.copy()

    # Get input and output dimensions
    input_dim = x_data.shape[1]
    output_dim = y_data.shape[1]
    print(f"Input dim: {input_dim}, Output dim: {output_dim}")

    # Store results for each fold
    fold_results = []
    all_test_predictions = []
    all_test_targets = []

    for test_group in range(5):  # 0-4 for 5 groups from KMeans
        print(f"\n{'='*50}")
        print(f"Fold {test_group}")
        print('='*50)
        
        # Get the split indexes for training and testing
        train_idx = cv_group_df['cv_group'] != test_group
        test_idx = cv_group_df['cv_group'] == test_group

        # Split the x and y data in train and test
        x_train = x_data.loc[train_idx]
        x_test = x_data.loc[test_idx]

        y_train = y_data.loc[train_idx]
        y_test = y_data.loc[test_idx]
        
        # Create train and validation data
        X_train, X_val, y_train, y_val = train_test_split(
            x_train, y_train, test_size=0.2, random_state=42
        )

        print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(x_test)}")

        # Scale the x data
        scaler = StandardScaler()
        X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
        X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns, index=X_val.index)
        x_test_scaled = pd.DataFrame(scaler.transform(x_test), columns=x_test.columns, index=x_test.index)

        #scale the y data
        y_scaler = StandardScaler()
        y_train_scaled = pd.DataFrame(y_scaler.fit_transform(y_train), columns=y_train.columns, index=y_train.index)
        y_val_scaled = pd.DataFrame(y_scaler.transform(y_val), columns=y_val.columns, index=y_val.index)
        y_test_scaled = pd.DataFrame(y_scaler.transform(y_test), columns=y_test.columns, index=y_test.index)

        # Convert DataFrames to PyTorch tensors
        X_train_t = torch.FloatTensor(X_train_scaled.values)
        y_train_t = torch.FloatTensor(y_train_scaled.values)
        X_val_t = torch.FloatTensor(X_val_scaled.values)
        y_val_t = torch.FloatTensor(y_val_scaled.values)
        X_test_t = torch.FloatTensor(x_test_scaled.values)
        y_test_t = torch.FloatTensor(y_test_scaled.values)

        # Create TensorDatasets
        train_dataset = TensorDataset(X_train_t, y_train_t)
        val_dataset = TensorDataset(X_val_t, y_val_t)
        test_dataset = TensorDataset(X_test_t, y_test_t)

        # Create DataLoaders
        # drop_last=True for train_loader to avoid batch size of 1 (causes BatchNorm to fail)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

        # Create the model with output_columns to auto-detect NF columns for softmax
        model = FlexibleNN(
            input_dim=input_dim,
            output_dim=output_dim,
            hidden_layers=hidden_layers,
            dropout_rate=dropout_rate,
            dropout_type=dropout_type,
            activation=activation,
            use_batch_norm=use_batch_norm,
            use_layer_norm=use_layer_norm,
            weight_decay=weight_decay,
            output_columns=y_data.columns.tolist()  # Auto-detect NF columns for softmax constraint
        ).to(device)

        # Get optimizer and criterion
        optimizer = model.get_optimizer(optimizer_type=optimizer_type, lr=lr)
        criterion = nn.MSELoss()

        # Training loop
        best_val_loss = float('inf')
        best_model_state = None
        patience_counter = 0
        train_losses = []
        val_losses = []

        for epoch in range(epochs):
            # Train
            train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
            
            # Evaluate on validation
            val_loss, _, _ = evaluate_epoch(model, val_loader, criterion, device)
            
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            
            # Track best model
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience_counter = 0
            else:
                patience_counter += 1
            
            # Verbose output
            if verbose and (epoch % 20 == 0 or epoch == epochs - 1):
                print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.6f} - Val Loss: {val_loss:.6f}")
            
            # Early stopping
            if early_stopping_patience and patience_counter >= early_stopping_patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

        # Restore best model
        if best_model_state is not None:
            model.load_state_dict(best_model_state)
            model.to(device)

        # Evaluate on test set (scaled)
        test_loss_scaled, test_preds_scaled, test_targets_scaled = evaluate_epoch(model, test_loader, criterion, device)
        
        # Inverse transform predictions and targets to original scale
        test_preds_original = y_scaler.inverse_transform(test_preds_scaled.numpy())
        test_targets_original = y_scaler.inverse_transform(test_targets_scaled.numpy())
        
        # Get column names for separating DF (Driving Force) vs NF (Phase Fraction)
        col_names = y_data.columns.tolist()
        df_cols = [i for i, col in enumerate(col_names) if col.startswith('DF_')]
        nf_cols = [i for i, col in enumerate(col_names) if col.startswith('NF_')]
        
        # Calculate overall RMSE and MAE on original scale (all values)
        mse_original = np.mean((test_preds_original - test_targets_original) ** 2)
        rmse = np.sqrt(mse_original)
        mae = np.mean(np.abs(test_preds_original - test_targets_original))
        
        # Calculate metrics for non-zero targets only
        # Use a small threshold to handle floating point precision
        nonzero_threshold = 1e-6
        
        # Overall non-zero metrics
        nonzero_mask = np.abs(test_targets_original) > nonzero_threshold
        if np.any(nonzero_mask):
            nonzero_errors = test_preds_original[nonzero_mask] - test_targets_original[nonzero_mask]
            rmse_nonzero = np.sqrt(np.mean(nonzero_errors ** 2))
            mae_nonzero = np.mean(np.abs(nonzero_errors))
            n_nonzero = np.sum(nonzero_mask)
        else:
            rmse_nonzero, mae_nonzero, n_nonzero = 0.0, 0.0, 0
        
        # Calculate metrics for Driving Force columns (DF_) - non-zero only
        if df_cols:
            df_preds = test_preds_original[:, df_cols]
            df_targets = test_targets_original[:, df_cols]
            df_nonzero_mask = np.abs(df_targets) > nonzero_threshold
            if np.any(df_nonzero_mask):
                df_errors = df_preds[df_nonzero_mask] - df_targets[df_nonzero_mask]
                df_rmse = np.sqrt(np.mean(df_errors ** 2))
                df_mae = np.mean(np.abs(df_errors))
                n_df_nonzero = np.sum(df_nonzero_mask)
            else:
                df_rmse, df_mae, n_df_nonzero = 0.0, 0.0, 0
        else:
            df_rmse, df_mae, n_df_nonzero = 0.0, 0.0, 0
        
        # Calculate metrics for Phase Fraction columns (NF_) - non-zero only
        if nf_cols:
            nf_preds = test_preds_original[:, nf_cols]
            nf_targets = test_targets_original[:, nf_cols]
            nf_nonzero_mask = np.abs(nf_targets) > nonzero_threshold
            if np.any(nf_nonzero_mask):
                nf_errors = nf_preds[nf_nonzero_mask] - nf_targets[nf_nonzero_mask]
                nf_rmse = np.sqrt(np.mean(nf_errors ** 2))
                nf_mae = np.mean(np.abs(nf_errors))
                n_nf_nonzero = np.sum(nf_nonzero_mask)
            else:
                nf_rmse, nf_mae, n_nf_nonzero = 0.0, 0.0, 0
        else:
            nf_rmse, nf_mae, n_nf_nonzero = 0.0, 0.0, 0
        
        print(f"Overall (all)      - RMSE: {rmse:.4f}, MAE: {mae:.4f}")
        print(f"Overall (non-zero) - RMSE: {rmse_nonzero:.4f}, MAE: {mae_nonzero:.4f}  [{n_nonzero:,} values]")
        print(f"Driving Force (DF) - RMSE: {df_rmse:.4f}, MAE: {df_mae:.4f}  [{n_df_nonzero:,} non-zero values]")
        print(f"Phase Fraction (NF) - RMSE: {nf_rmse:.4f}, MAE: {nf_mae:.4f}  [{n_nf_nonzero:,} non-zero values]")

        # Store fold results
        fold_results.append({
            'fold': test_group,
            'best_val_loss': best_val_loss,
            'test_loss': mse_original,
            'test_rmse': rmse,
            'test_mae': mae,
            'rmse_nonzero': rmse_nonzero,
            'mae_nonzero': mae_nonzero,
            'df_rmse': df_rmse,
            'df_mae': df_mae,
            'nf_rmse': nf_rmse,
            'nf_mae': nf_mae,
            'train_losses': train_losses,
            'val_losses': val_losses
        })
        
        all_test_predictions.append(torch.FloatTensor(test_preds_original))
        all_test_targets.append(torch.FloatTensor(test_targets_original))

    # Summary across all folds
    print(f"\n{'='*50}")
    print("Cross-Validation Summary (non-zero targets only)")
    print('='*50)
    avg_test_loss = np.mean([r['test_loss'] for r in fold_results])
    avg_test_rmse = np.mean([r['test_rmse'] for r in fold_results])
    avg_test_mae = np.mean([r['test_mae'] for r in fold_results])
    avg_rmse_nonzero = np.mean([r['rmse_nonzero'] for r in fold_results])
    avg_mae_nonzero = np.mean([r['mae_nonzero'] for r in fold_results])
    std_test_loss = np.std([r['test_loss'] for r in fold_results])
    
    # Separate metrics for DF and NF (already non-zero)
    avg_df_rmse = np.mean([r['df_rmse'] for r in fold_results])
    avg_df_mae = np.mean([r['df_mae'] for r in fold_results])
    avg_nf_rmse = np.mean([r['nf_rmse'] for r in fold_results])
    avg_nf_mae = np.mean([r['nf_mae'] for r in fold_results])
    
    print(f"Overall (all values):")
    print(f"  Avg Test MSE: {avg_test_loss:.4f} (+/- {std_test_loss:.4f})")
    print(f"  Avg RMSE: {avg_test_rmse:.4f}, MAE: {avg_test_mae:.4f}")
    print(f"\nOverall (non-zero only):")
    print(f"  Avg RMSE: {avg_rmse_nonzero:.4f}, MAE: {avg_mae_nonzero:.4f}")
    print(f"\nDriving Force (DF, non-zero only):")
    print(f"  Avg RMSE: {avg_df_rmse:.4f}, MAE: {avg_df_mae:.4f}")
    print(f"\nPhase Fraction (NF, non-zero only):")
    print(f"  Avg RMSE: {avg_nf_rmse:.4f}, MAE: {avg_nf_mae:.4f}")

    return {
        'fold_results': fold_results,
        'avg_test_loss': avg_test_loss,
        'avg_test_rmse': avg_test_rmse,
        'avg_test_mae': avg_test_mae,
        'avg_rmse_nonzero': avg_rmse_nonzero,
        'avg_mae_nonzero': avg_mae_nonzero,
        'avg_df_rmse': avg_df_rmse,
        'avg_df_mae': avg_df_mae,
        'avg_nf_rmse': avg_nf_rmse,
        'avg_nf_mae': avg_nf_mae,
        'all_predictions': all_test_predictions,
        'all_targets': all_test_targets
    }

In [13]:
def evaluate_parameters_DFnn(parameters, batch_size=32, epochs=1000, verbose=True):
    
    # Extract hyperparameters from parameters dict with defaults
    batch_size = parameters.get('batch_size', batch_size)
    hidden_layers = parameters.get('hidden_layers', [256, 128, 64])
    dropout_rate = parameters.get('dropout_rate', 0.2)
    dropout_type = parameters.get('dropout_type', 'standard')
    activation = parameters.get('activation', 'relu')
    use_batch_norm = parameters.get('use_batch_norm', False)
    use_layer_norm = parameters.get('use_layer_norm', False)
    weight_decay = parameters.get('weight_decay', 0.0)
    optimizer_type = parameters.get('optimizer_type', 'adam')
    lr = parameters.get('lr', 1e-3)
    early_stopping_patience = parameters.get('early_stopping_patience', 10)

    # Auto-detect device
    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device('cpu')
    print(f"Using device: {device}")

    # Copy the x and y data — use driving force targets from cell 11
    y_data = y_df.copy()
    x_data = X_combined.copy()

    # Get input and output dimensions
    input_dim = x_data.shape[1]
    output_dim = y_data.shape[1]
    print(f"Input dim: {input_dim}, Output dim: {output_dim}")

    # Store results for each fold
    fold_results = []
    all_test_predictions = []
    all_test_targets = []

    for test_group in range(5):  # 0-4 for 5 groups from KMeans
        print(f"\n{'='*50}")
        print(f"Fold {test_group}")
        print('='*50)
        
        # Get the split indexes for training and testing
        train_idx = cv_group_df['cv_group'] != test_group
        test_idx = cv_group_df['cv_group'] == test_group

        # Split the x and y data in train and test
        x_train = x_data.loc[train_idx]
        x_test = x_data.loc[test_idx]

        y_train = y_data.loc[train_idx]
        y_test = y_data.loc[test_idx]
        
        # Create train and validation data
        X_train, X_val, y_train, y_val = train_test_split(
            x_train, y_train, test_size=0.2, random_state=42
        )

        print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(x_test)}")

        # Scale the x data
        scaler = StandardScaler()
        X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
        X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns, index=X_val.index)
        x_test_scaled = pd.DataFrame(scaler.transform(x_test), columns=x_test.columns, index=x_test.index)

        # Scale the y data
        y_scaler = StandardScaler()
        y_train_scaled = pd.DataFrame(y_scaler.fit_transform(y_train), columns=y_train.columns, index=y_train.index)
        y_val_scaled = pd.DataFrame(y_scaler.transform(y_val), columns=y_val.columns, index=y_val.index)
        y_test_scaled = pd.DataFrame(y_scaler.transform(y_test), columns=y_test.columns, index=y_test.index)

        # Convert DataFrames to PyTorch tensors
        X_train_t = torch.FloatTensor(X_train_scaled.values)
        y_train_t = torch.FloatTensor(y_train_scaled.values)
        X_val_t = torch.FloatTensor(X_val_scaled.values)
        y_val_t = torch.FloatTensor(y_val_scaled.values)
        X_test_t = torch.FloatTensor(x_test_scaled.values)
        y_test_t = torch.FloatTensor(y_test_scaled.values)

        # Create TensorDatasets
        train_dataset = TensorDataset(X_train_t, y_train_t)
        val_dataset = TensorDataset(X_val_t, y_val_t)
        test_dataset = TensorDataset(X_test_t, y_test_t)

        # Create DataLoaders
        # drop_last=True for train_loader to avoid batch size of 1 (causes BatchNorm to fail)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

        # Create the DrivingForceNN model (no softmax — outputs are unconstrained real values)
        model = DrivingForceNN(
            input_dim=input_dim,
            output_dim=output_dim,
            hidden_layers=hidden_layers,
            dropout_rate=dropout_rate,
            dropout_type=dropout_type,
            activation=activation,
            use_batch_norm=use_batch_norm,
            use_layer_norm=use_layer_norm,
            weight_decay=weight_decay,
        ).to(device)

        # Get optimizer and criterion
        optimizer = model.get_optimizer(optimizer_type=optimizer_type, lr=lr)
        criterion = nn.MSELoss()

        # Training loop
        best_val_loss = float('inf')
        best_model_state = None
        patience_counter = 0
        train_losses = []
        val_losses = []

        for epoch in range(epochs):
            # Train
            train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
            
            # Evaluate on validation
            val_loss, _, _ = evaluate_epoch(model, val_loader, criterion, device)
            
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            
            # Track best model
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience_counter = 0
            else:
                patience_counter += 1
            
            # Verbose output
            if verbose and (epoch % 20 == 0 or epoch == epochs - 1):
                print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.6f} - Val Loss: {val_loss:.6f}")
            
            # Early stopping
            if early_stopping_patience and patience_counter >= early_stopping_patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

        # Restore best model
        if best_model_state is not None:
            model.load_state_dict(best_model_state)
            model.to(device)

        # Evaluate on test set (scaled)
        test_loss_scaled, test_preds_scaled, test_targets_scaled = evaluate_epoch(model, test_loader, criterion, device)
        
        # Inverse transform predictions and targets to original scale
        test_preds_original = y_scaler.inverse_transform(test_preds_scaled.numpy())
        test_targets_original = y_scaler.inverse_transform(test_targets_scaled.numpy())
        
        # Calculate overall RMSE and MAE on original scale (all values)
        mse_original = np.mean((test_preds_original - test_targets_original) ** 2)
        rmse = np.sqrt(mse_original)
        mae = np.mean(np.abs(test_preds_original - test_targets_original))
        
        # Calculate metrics for non-zero targets only
        nonzero_threshold = 1e-6
        
        nonzero_mask = np.abs(test_targets_original) > nonzero_threshold
        if np.any(nonzero_mask):
            nonzero_errors = test_preds_original[nonzero_mask] - test_targets_original[nonzero_mask]
            rmse_nonzero = np.sqrt(np.mean(nonzero_errors ** 2))
            mae_nonzero = np.mean(np.abs(nonzero_errors))
            n_nonzero = np.sum(nonzero_mask)
        else:
            rmse_nonzero, mae_nonzero, n_nonzero = 0.0, 0.0, 0
        
        # Per-column RMSE for driving force outputs (non-zero only)
        col_names = y_data.columns.tolist()
        per_col_rmse = {}
        for ci, col in enumerate(col_names):
            col_mask = np.abs(test_targets_original[:, ci]) > nonzero_threshold
            if np.any(col_mask):
                col_errors = test_preds_original[:, ci][col_mask] - test_targets_original[:, ci][col_mask]
                per_col_rmse[col] = np.sqrt(np.mean(col_errors ** 2))
        
        print(f"Driving Force (all)      - RMSE: {rmse:.4f}, MAE: {mae:.4f}")
        print(f"Driving Force (non-zero) - RMSE: {rmse_nonzero:.4f}, MAE: {mae_nonzero:.4f}  [{n_nonzero:,} values]")

        # Store fold results
        fold_results.append({
            'fold': test_group,
            'best_val_loss': best_val_loss,
            'test_loss': mse_original,
            'test_rmse': rmse,
            'test_mae': mae,
            'rmse_nonzero': rmse_nonzero,
            'mae_nonzero': mae_nonzero,
            'per_col_rmse': per_col_rmse,
            'train_losses': train_losses,
            'val_losses': val_losses
        })
        
        all_test_predictions.append(torch.FloatTensor(test_preds_original))
        all_test_targets.append(torch.FloatTensor(test_targets_original))

    # Summary across all folds
    print(f"\n{'='*50}")
    print("Cross-Validation Summary — Driving Forces")
    print('='*50)
    avg_test_loss = np.mean([r['test_loss'] for r in fold_results])
    avg_test_rmse = np.mean([r['test_rmse'] for r in fold_results])
    avg_test_mae = np.mean([r['test_mae'] for r in fold_results])
    avg_rmse_nonzero = np.mean([r['rmse_nonzero'] for r in fold_results])
    avg_mae_nonzero = np.mean([r['mae_nonzero'] for r in fold_results])
    std_test_loss = np.std([r['test_loss'] for r in fold_results])
    
    print(f"Driving Force (all values):")
    print(f"  Avg Test MSE: {avg_test_loss:.4f} (+/- {std_test_loss:.4f})")
    print(f"  Avg RMSE: {avg_test_rmse:.4f}, MAE: {avg_test_mae:.4f}")
    print(f"\nDriving Force (non-zero only):")
    print(f"  Avg RMSE: {avg_rmse_nonzero:.4f}, MAE: {avg_mae_nonzero:.4f}")

    return {
        'fold_results': fold_results,
        'avg_test_loss': avg_test_loss,
        'avg_test_rmse': avg_test_rmse,
        'avg_test_mae': avg_test_mae,
        'avg_rmse_nonzero': avg_rmse_nonzero,
        'avg_mae_nonzero': avg_mae_nonzero,
        'all_predictions': all_test_predictions,
        'all_targets': all_test_targets
    }

In [ ]:
#test evaluate parameters for NFnn
parameters = {
    'hidden_layers': [256, 512, 1024],
    'activation': 'gelu',
    'use_batch_norm': True,
    'lr': 1e-3,
    'early_stopping_patience': 15
}
results = evaluate_parameters_NFnn(parameters)

Using device: mps
Input dim: 144, Output dim: 804

Fold 0
Train: 29906, Val: 7477, Test: 7599
Epoch 1/1000 - Train Loss: 0.255452 - Val Loss: 0.344018
Epoch 21/1000 - Train Loss: 0.252722 - Val Loss: 0.341859
Epoch 41/1000 - Train Loss: 0.252290 - Val Loss: 0.341576
Epoch 61/1000 - Train Loss: 0.252245 - Val Loss: 0.341517
Epoch 81/1000 - Train Loss: 0.251786 - Val Loss: 0.341334
Epoch 101/1000 - Train Loss: 0.252075 - Val Loss: 0.341304
Epoch 121/1000 - Train Loss: 0.251710 - Val Loss: 0.341210
Epoch 141/1000 - Train Loss: 0.251468 - Val Loss: 0.341111
Epoch 161/1000 - Train Loss: 0.251638 - Val Loss: 0.341095
Early stopping at epoch 163
Overall (all)      - RMSE: 0.0366, MAE: 0.0023
Overall (non-zero) - RMSE: 0.4589, MAE: 0.3844  [11,193 values]
Driving Force (DF) - RMSE: 0.0000, MAE: 0.0000  [0 non-zero values]
Phase Fraction (NF) - RMSE: 0.4589, MAE: 0.3844  [11,193 non-zero values]

Fold 1
Train: 23174, Val: 5794, Test: 16014
Epoch 1/1000 - Train Loss: 0.237284 - Val Loss: 0.26631

KeyboardInterrupt: 

In [21]:
# Define architecture presets mapping (all expanding for large output dim)
ARCHITECTURE_PRESETS = {
    # Single layer - direct expansion
    "single_256": [256],
    "single_512": [512],
    "single_1024": [1024],
    "single_2048": [2048],
    # Two layers - expanding
    "expand_2L_small": [256, 512],
    "expand_2L_medium": [512, 1024],
    "expand_2L_large": [1024, 2048],
    "expand_2L_xlarge": [512, 2048],
    # Three layers - expanding
    "expand_3L_small": [256, 512, 1024],
    "expand_3L_medium": [512, 1024, 2048],
    "expand_3L_large": [256, 1024, 2048],
    "expand_3L_gradual": [384, 768, 1536],
    # Four layers - expanding
    "expand_4L_small": [256, 512, 1024, 2048],
    "expand_4L_medium": [512, 768, 1024, 2048],
    "expand_4L_large": [256, 512, 1024, 4096],
    # Constant width (also good for large outputs)
    "constant_512": [512, 512],
    "constant_1024": [1024, 1024],
    "constant_2048": [2048, 2048],
    "constant_1024_3L": [1024, 1024, 1024],
}

ax_client = AxClient()
ax_client.create_experiment(
    name="NN opt NFnn f(T) Calphed",
    parameters=[
        {
            "name": "architecture",
            "type": "choice",
            "values": list(ARCHITECTURE_PRESETS.keys()),
            "is_ordered": False,
        },
        {
            "name": "dropout_rate",
            "type": "range",
            "bounds": [0.0, 0.5],
        },
        {
            "name": "activation",
            "type": "choice",
            "values": ["relu", "leaky_relu", "elu", "selu", "gelu"],
        },
        {
            "name": "normalization",
            "type": "choice",
            "values": ["none", "batch_norm", "layer_norm"],
        },
        {
            "name": "weight_decay",
            "type": "range",
            "bounds": [1e-6, 1e-2],
            "log_scale": True,
        },
        {
            "name": "lr",
            "type": "range",
            "bounds": [1e-5, 1e-2],
            "log_scale": True,
        },
        {
            "name": "optimizer_type",
            "type": "choice",
            "values": ["adam", "adamw"],
        },
        {
            "name": "batch_size",
            "type": "choice",
            "values": [32, 64, 128, 256],
        },
        {
            "name": "early_stopping_patience",
            "type": "range",
            "bounds": [10, 50],
            "value_type": "int",
        },
    ],
    objectives={"avg_rmse_nonzero": ObjectiveProperties(minimize=True)}
)

def evaluate_for_ax(parameterization):
    """Wrapper to convert Ax parameters to evaluate_parameters_NFnn format."""
    # Get hidden_layers from architecture preset
    architecture = parameterization["architecture"]
    hidden_layers = ARCHITECTURE_PRESETS[architecture]
    
    # Handle normalization choice
    normalization = parameterization["normalization"]
    use_batch_norm = normalization == "batch_norm"
    use_layer_norm = normalization == "layer_norm"
    
    # Handle dropout type based on activation
    activation = parameterization["activation"]
    dropout_type = "alpha" if activation == "selu" else "standard"
    
    # Build parameters dict
    parameters = {
        "hidden_layers": hidden_layers,
        "dropout_rate": parameterization["dropout_rate"],
        "dropout_type": dropout_type,
        "activation": activation,
        "use_batch_norm": use_batch_norm,
        "use_layer_norm": use_layer_norm,
        "weight_decay": parameterization["weight_decay"],
        "lr": parameterization["lr"],
        "optimizer_type": parameterization["optimizer_type"],
        "batch_size": parameterization["batch_size"],
        "early_stopping_patience": parameterization["early_stopping_patience"],
    }
    
    # Run evaluation using NFnn function
    results = evaluate_parameters_NFnn(parameters, verbose=False)
    
    # Calculate SEM (Standard Error of the Mean) from fold results
    # SEM = std / sqrt(n_folds)
    rmse_nonzero_values = [r["rmse_nonzero"] for r in results["fold_results"]]
    mean_rmse_nonzero = np.mean(rmse_nonzero_values)
    std_rmse_nonzero = np.std(rmse_nonzero_values, ddof=1)  # ddof=1 for sample std
    sem_rmse_nonzero = std_rmse_nonzero / np.sqrt(len(rmse_nonzero_values))
    
    # Return the objective with proper SEM for Bayesian optimization
    return {"avg_rmse_nonzero": (mean_rmse_nonzero, sem_rmse_nonzero)}

[INFO 02-11 08:57:18] ax.service.ax_client: Starting optimization with verbose logging. To disable logging, set the `verbose_logging` argument to `False`. Note that float values in the logs are rounded to 6 decimal points.
[INFO 02-11 08:57:18] ax.service.utils.instantiation: Inferred value type of ParameterType.STRING for parameter architecture. If that is not the expected value type, you can explicitly specify 'value_type' ('int', 'float', 'bool' or 'str') in parameter dict.
c:\Users\Chris\pytorch_gpu\Lib\site-packages\ax\service\utils\instantiation.py:258: AxParameterWarning: `sort_values` is not specified for `ChoiceParameter` "architecture". Defaulting to `False` for parameters of `ParameterType` STRING. To override this behavior (or avoid this warning), specify `sort_values` during `ChoiceParameter` construction.
  return ChoiceParameter(
[INFO 02-11 08:57:18] ax.service.utils.instantiation: Inferred value type of ParameterType.FLOAT for parameter dropout_rate. If that is not the

In [ ]:
# Run the Bayesian optimization loop
target_trials= 100
SAVE_PATH = r"Ax_checkpoints/ax_client_NN_NF(T)_checkpoint.json"

#load the partially completed AxClient from the checkpoint file and continue
ax_client = AxClient().load_from_json_file(SAVE_PATH)
completed_trials = len(ax_client.experiment.trials)

remaining_trials = target_trials - completed_trials

print(f"Loaded {completed_trials} completed trials from checkpoint")
print(f"Remaining trials to reach {target_trials}: {remaining_trials}")

if remaining_trials > 0:
    for i in range(remaining_trials):
        current_trial = completed_trials + i + 1
        print(f"\n{'='*60}")
        print(f"Trial {current_trial}/{target_trials}")
        print('='*60)
        
        parameters, trial_index = ax_client.get_next_trial()
        print(f"Parameters: {parameters}")
        
        try:
            result = evaluate_for_ax(parameters)
            ax_client.complete_trial(trial_index=trial_index, raw_data=result)
            print(f"Result: Non-zero RMSE = {result['avg_rmse_nonzero'][0]:.4f}")
        except Exception as e:
            print(f"Trial failed: {e}")
            ax_client.log_trial_failure(trial_index=trial_index)
        
        # Save checkpoint after each trial
        ax_client.save_to_json_file(SAVE_PATH)
        print(f"Checkpoint saved to {SAVE_PATH}")

# Get best parameters
best_parameters, values = ax_client.get_best_parameters()
print(f"\n{'='*60}")
print("OPTIMIZATION COMPLETE")
print('='*60)
print(f"Best parameters: {best_parameters}")
print(f"Best Non-zero RMSE: {values[0]['avg_rmse_nonzero']:.4f}")

c:\Users\Chris\pytorch_gpu\Lib\site-packages\ax\modelbridge\cross_validation.py:439: UserWarning: Encountered exception in computing model fit quality: RandomModelBridge does not support prediction.
  warn("Encountered exception in computing model fit quality: " + str(e))
[INFO 02-11 08:57:24] ax.service.ax_client: Generated new trial 69 with parameters {'dropout_rate': 0.236128, 'weight_decay': 5e-06, 'lr': 0.005613, 'optimizer_type': 'adamw', 'batch_size': 256, 'early_stopping_patience': 39, 'architecture': 'single_256', 'activation': 'relu', 'normalization': 'none'} using model Sobol.



Trial 1/32
Parameters: {'dropout_rate': 0.2361278384923935, 'weight_decay': 5.245494132389832e-06, 'lr': 0.0056128659217602826, 'optimizer_type': 'adamw', 'batch_size': 256, 'early_stopping_patience': 39, 'architecture': 'single_256', 'activation': 'relu', 'normalization': 'none'}
Using device: cuda
Input dim: 144, Output dim: 804

Fold 0
Train: 29906, Val: 7477, Test: 7599
Early stopping at epoch 50
Overall (all)      - RMSE: 0.0251, MAE: 0.0014
Overall (non-zero) - RMSE: 0.4624, MAE: 0.3642  [11,193 values]
Driving Force (DF) - RMSE: 0.0000, MAE: 0.0000  [0 non-zero values]
Phase Fraction (NF) - RMSE: 0.4624, MAE: 0.3642  [11,193 non-zero values]

Fold 1
Train: 23174, Val: 5794, Test: 16014
Early stopping at epoch 106
Overall (all)      - RMSE: 0.0241, MAE: 0.0014
Overall (non-zero) - RMSE: 0.3865, MAE: 0.2917  [27,977 values]
Driving Force (DF) - RMSE: 0.0000, MAE: 0.0000  [0 non-zero values]
Phase Fraction (NF) - RMSE: 0.3865, MAE: 0.2917  [27,977 non-zero values]

Fold 2
Train: 3

[INFO 02-11 09:03:06] ax.service.ax_client: Completed trial 69 with data: {'avg_rmse_nonzero': (np.float32(0.39311767), np.float64(0.026503))}.
[INFO 02-11 09:03:06] ax.service.ax_client: Saved JSON-serialized state of optimization to `c:\Users\Chris\OneDrive\Desktop\ML Project\ax_client_Phase(T)_checkpoint.json`.
c:\Users\Chris\pytorch_gpu\Lib\site-packages\ax\modelbridge\cross_validation.py:439: UserWarning: Encountered exception in computing model fit quality: RandomModelBridge does not support prediction.
  warn("Encountered exception in computing model fit quality: " + str(e))
[INFO 02-11 09:03:06] ax.service.ax_client: Generated new trial 70 with parameters {'dropout_rate': 0.363807, 'weight_decay': 0.00117, 'lr': 8.6e-05, 'optimizer_type': 'adam', 'batch_size': 64, 'early_stopping_patience': 23, 'architecture': 'single_512', 'activation': 'gelu', 'normalization': 'batch_norm'} using model Sobol.


Result: Non-zero RMSE = 0.3931
Checkpoint saved to c:\Users\Chris\OneDrive\Desktop\ML Project\ax_client_Phase(T)_checkpoint.json

Trial 2/32
Parameters: {'dropout_rate': 0.3638072586618364, 'weight_decay': 0.0011697226884563882, 'lr': 8.599836076315322e-05, 'optimizer_type': 'adam', 'batch_size': 64, 'early_stopping_patience': 23, 'architecture': 'single_512', 'activation': 'gelu', 'normalization': 'batch_norm'}
Using device: cuda
Input dim: 144, Output dim: 804

Fold 0
Train: 29906, Val: 7477, Test: 7599
Early stopping at epoch 28
Overall (all)      - RMSE: 0.0245, MAE: 0.0024
Overall (non-zero) - RMSE: 0.4907, MAE: 0.4204  [11,193 values]
Driving Force (DF) - RMSE: 0.0000, MAE: 0.0000  [0 non-zero values]
Phase Fraction (NF) - RMSE: 0.4907, MAE: 0.4204  [11,193 non-zero values]

Fold 1
Train: 23174, Val: 5794, Test: 16014
Early stopping at epoch 30
Overall (all)      - RMSE: 0.0236, MAE: 0.0024
Overall (non-zero) - RMSE: 0.4119, MAE: 0.3323  [27,977 values]
Driving Force (DF) - RMSE:

[INFO 02-11 09:06:25] ax.service.ax_client: Completed trial 70 with data: {'avg_rmse_nonzero': (np.float32(0.42817968), np.float64(0.027335))}.
[INFO 02-11 09:06:25] ax.service.ax_client: Saved JSON-serialized state of optimization to `c:\Users\Chris\OneDrive\Desktop\ML Project\ax_client_Phase(T)_checkpoint.json`.
c:\Users\Chris\pytorch_gpu\Lib\site-packages\ax\modelbridge\cross_validation.py:439: UserWarning: Encountered exception in computing model fit quality: RandomModelBridge does not support prediction.
  warn("Encountered exception in computing model fit quality: " + str(e))
[INFO 02-11 09:06:25] ax.service.ax_client: Generated new trial 71 with parameters {'dropout_rate': 0.107884, 'weight_decay': 7.4e-05, 'lr': 1.5e-05, 'optimizer_type': 'adamw', 'batch_size': 128, 'early_stopping_patience': 43, 'architecture': 'expand_4L_large', 'activation': 'leaky_relu', 'normalization': 'layer_norm'} using model Sobol.


...

Overall (all)      - RMSE: 0.0163, MAE: 0.0010
Overall (non-zero) - RMSE: 0.3139, MAE: 0.2274  [6,176 values]
Driving Force (DF) - RMSE: 0.0000, MAE: 0.0000  [0 non-zero values]
Phase Fraction (NF) - RMSE: 0.3139, MAE: 0.2274  [6,176 non-zero values]

Cross-Validation Summary (non-zero targets only)
Overall (all values):
  Avg Test MSE: 0.0006 (+/- 0.0002)
  Avg RMSE: 0.0241, MAE: 0.0014

Overall (non-zero only):
  Avg RMSE: 0.3822, MAE: 0.2869

Driving Force (DF, non-zero only):
  Avg RMSE: 0.0000, MAE: 0.0000

Phase Fraction (NF, non-zero only):
  Avg RMSE: 0.3822, MAE: 0.2869
Result: Non-zero RMSE = 0.3822
Checkpoint saved to c:\Users\Chris\OneDrive\Desktop\ML Project\ax_client_Phase(T)_checkpoint.json

Trial 32/32


c:\Users\Chris\pytorch_gpu\Lib\site-packages\botorch\optim\optimize_mixed.py:702: OptimizationWarning: Failed to initialize using continuous relaxation. Using `sample_feasible_points` for initialization. Original error message: '0.9918767776527053'
  best_X, best_acq_val = generate_starting_points(
C:\Users\Chris\python\Lib\dataclasses.py:1581: RuntimeWarning: If using a 2-dim `batch_initial_conditions` botorch will default to old behavior of ignoring `num_restarts` and just use the given `batch_initial_conditions` by setting `raw_samples` to None.
  return obj.__class__(**changes)
C:\Users\Chris\python\Lib\dataclasses.py:1581: RuntimeWarning: If using a 2-dim `batch_initial_conditions` botorch will default to old behavior of ignoring `num_restarts` and just use the given `batch_initial_conditions` by setting `raw_samples` to None.
  return obj.__class__(**changes)
C:\Users\Chris\python\Lib\dataclasses.py:1581: RuntimeWarning: If using a 2-dim `batch_initial_conditions` botorch will de

Parameters: {'dropout_rate': 0.5, 'weight_decay': 9.088122501327628e-05, 'lr': 1e-05, 'optimizer_type': 'adamw', 'batch_size': 32, 'early_stopping_patience': 48, 'architecture': 'constant_1024_3L', 'activation': 'leaky_relu', 'normalization': 'layer_norm'}
Using device: cuda
Input dim: 144, Output dim: 804

Fold 0
Train: 29906, Val: 7477, Test: 7599
Early stopping at epoch 749
Overall (all)      - RMSE: 0.0271, MAE: 0.0014
Overall (non-zero) - RMSE: 0.4377, MAE: 0.3165  [11,193 values]
Driving Force (DF) - RMSE: 0.0000, MAE: 0.0000  [0 non-zero values]
Phase Fraction (NF) - RMSE: 0.4377, MAE: 0.3165  [11,193 non-zero values]

Fold 1
Train: 23174, Val: 5794, Test: 16014
Early stopping at epoch 472
Overall (all)      - RMSE: 0.0247, MAE: 0.0015
Overall (non-zero) - RMSE: 0.3735, MAE: 0.2774  [27,977 values]
Driving Force (DF) - RMSE: 0.0000, MAE: 0.0000  [0 non-zero values]
Phase Fraction (NF) - RMSE: 0.3735, MAE: 0.2774  [27,977 non-zero values]

Fold 2
Train: 30885, Val: 7722, Test: 63

[INFO 02-15 06:01:21] ax.service.ax_client: Completed trial 100 with data: {'avg_rmse_nonzero': (np.float32(0.36970183), np.float64(0.033123))}.
[INFO 02-15 06:01:21] ax.service.ax_client: Saved JSON-serialized state of optimization to `c:\Users\Chris\OneDrive\Desktop\ML Project\ax_client_Phase(T)_checkpoint.json`.


Overall (all)      - RMSE: 0.0168, MAE: 0.0011
Overall (non-zero) - RMSE: 0.2993, MAE: 0.2162  [6,176 values]
Driving Force (DF) - RMSE: 0.0000, MAE: 0.0000  [0 non-zero values]
Phase Fraction (NF) - RMSE: 0.2993, MAE: 0.2162  [6,176 non-zero values]

Cross-Validation Summary (non-zero targets only)
Overall (all values):
  Avg Test MSE: 0.0006 (+/- 0.0002)
  Avg RMSE: 0.0245, MAE: 0.0014

Overall (non-zero only):
  Avg RMSE: 0.3697, MAE: 0.2703

Driving Force (DF, non-zero only):
  Avg RMSE: 0.0000, MAE: 0.0000

Phase Fraction (NF, non-zero only):
  Avg RMSE: 0.3697, MAE: 0.2703
Result: Non-zero RMSE = 0.3697
Checkpoint saved to c:\Users\Chris\OneDrive\Desktop\ML Project\ax_client_Phase(T)_checkpoint.json

OPTIMIZATION COMPLETE
Best parameters: {'dropout_rate': 0.5, 'weight_decay': 1e-06, 'lr': 1e-05, 'optimizer_type': 'adamw', 'batch_size': 32, 'early_stopping_patience': 50, 'architecture': 'constant_1024_3L', 'activation': 'relu', 'normalization': 'batch_norm'}
Best Non-zero RMSE: 0.

In [ ]:
# =====================================================================
# Ax Bayesian Optimization for Driving Force NN
# =====================================================================

# Reuse the same architecture presets
ARCHITECTURE_PRESETS_DF = {
    # Single layer
    "single_256": [256],
    "single_512": [512],
    "single_1024": [1024],
    "single_2048": [2048],
    # Two layers - expanding
    "expand_2L_small": [256, 512],
    "expand_2L_medium": [512, 1024],
    "expand_2L_large": [1024, 2048],
    "expand_2L_xlarge": [512, 2048],
    # Three layers - expanding
    "expand_3L_small": [256, 512, 1024],
    "expand_3L_medium": [512, 1024, 2048],
    "expand_3L_large": [256, 1024, 2048],
    "expand_3L_gradual": [384, 768, 1536],
    # Four layers - expanding
    "expand_4L_small": [256, 512, 1024, 2048],
    "expand_4L_medium": [512, 768, 1024, 2048],
    "expand_4L_large": [256, 512, 1024, 4096],
    # Constant width
    "constant_512": [512, 512],
    "constant_1024": [1024, 1024],
    "constant_2048": [2048, 2048],
    "constant_1024_3L": [1024, 1024, 1024],
}

ax_client_df = AxClient()
ax_client_df.create_experiment(
    name="NN opt DFnn f(T) Calphed",
    parameters=[
        {
            "name": "architecture",
            "type": "choice",
            "values": list(ARCHITECTURE_PRESETS_DF.keys()),
            "is_ordered": False,
        },
        {
            "name": "dropout_rate",
            "type": "range",
            "bounds": [0.0, 0.5],
        },
        {
            "name": "activation",
            "type": "choice",
            "values": ["relu", "leaky_relu", "elu", "selu", "gelu"],
        },
        {
            "name": "normalization",
            "type": "choice",
            "values": ["none", "batch_norm", "layer_norm"],
        },
        {
            "name": "weight_decay",
            "type": "range",
            "bounds": [1e-6, 1e-2],
            "log_scale": True,
        },
        {
            "name": "lr",
            "type": "range",
            "bounds": [1e-5, 1e-2],
            "log_scale": True,
        },
        {
            "name": "optimizer_type",
            "type": "choice",
            "values": ["adam", "adamw"],
        },
        {
            "name": "batch_size",
            "type": "choice",
            "values": [32, 64, 128, 256],
        },
        {
            "name": "early_stopping_patience",
            "type": "range",
            "bounds": [10, 50],
            "value_type": "int",
        },
    ],
    objectives={"avg_rmse_nonzero": ObjectiveProperties(minimize=True)}
)


def evaluate_for_ax_df(parameterization):
    """Wrapper to convert Ax parameters to evaluate_parameters_DFnn format."""
    # Get hidden_layers from architecture preset
    architecture = parameterization["architecture"]
    hidden_layers = ARCHITECTURE_PRESETS_DF[architecture]

    # Handle normalization choice
    normalization = parameterization["normalization"]
    use_batch_norm = normalization == "batch_norm"
    use_layer_norm = normalization == "layer_norm"

    # Handle dropout type based on activation
    activation = parameterization["activation"]
    dropout_type = "alpha" if activation == "selu" else "standard"

    # Build parameters dict
    parameters = {
        "hidden_layers": hidden_layers,
        "dropout_rate": parameterization["dropout_rate"],
        "dropout_type": dropout_type,
        "activation": activation,
        "use_batch_norm": use_batch_norm,
        "use_layer_norm": use_layer_norm,
        "weight_decay": parameterization["weight_decay"],
        "lr": parameterization["lr"],
        "optimizer_type": parameterization["optimizer_type"],
        "batch_size": parameterization["batch_size"],
        "early_stopping_patience": parameterization["early_stopping_patience"],
    }

    # Run evaluation using DFnn function
    results = evaluate_parameters_DFnn(parameters, verbose=False)

    # Calculate SEM (Standard Error of the Mean) from fold results
    rmse_nonzero_values = [r["rmse_nonzero"] for r in results["fold_results"]]
    mean_rmse_nonzero = np.mean(rmse_nonzero_values)
    std_rmse_nonzero = np.std(rmse_nonzero_values, ddof=1)
    sem_rmse_nonzero = std_rmse_nonzero / np.sqrt(len(rmse_nonzero_values))

    return {"avg_rmse_nonzero": (mean_rmse_nonzero, sem_rmse_nonzero)}


# =====================================================================
# Run the Bayesian optimization loop for Driving Force NN
# =====================================================================
n_trials_df = 32  # Adjust based on your time budget
SAVE_PATH_DF = r"Ax_checkpoints/ax_client_NN_DF(T)_checkpoint.json"

for i in range(n_trials_df):
    print(f"\n{'='*60}")
    print(f"Trial {i+1}/{n_trials_df}")
    print('='*60)

    parameters, trial_index = ax_client_df.get_next_trial()
    print(f"Parameters: {parameters}")

    try:
        result = evaluate_for_ax_df(parameters)
        ax_client_df.complete_trial(trial_index=trial_index, raw_data=result)
        print(f"Result: Non-zero RMSE = {result['avg_rmse_nonzero'][0]:.4f}")
    except Exception as e:
        print(f"Trial failed: {e}")
        ax_client_df.log_trial_failure(trial_index=trial_index)

    # Save checkpoint after each trial
    ax_client_df.save_to_json_file(SAVE_PATH_DF)
    print(f"Checkpoint saved to {SAVE_PATH_DF}")

# Get best parameters
best_parameters_df, values_df = ax_client_df.get_best_parameters()
print(f"\n{'='*60}")
print("OPTIMIZATION COMPLETE — Driving Force NN")
print('='*60)
print(f"Best parameters: {best_parameters_df}")
print(f"Best Non-zero RMSE: {values_df[0]['avg_rmse_nonzero']:.4f}")

[INFO 02-17 08:43:16] ax.service.ax_client: Starting optimization with verbose logging. To disable logging, set the `verbose_logging` argument to `False`. Note that float values in the logs are rounded to 6 decimal points.
[INFO 02-17 08:43:16] ax.service.utils.instantiation: Inferred value type of ParameterType.STRING for parameter architecture. If that is not the expected value type, you can explicitly specify 'value_type' ('int', 'float', 'bool' or 'str') in parameter dict.
c:\Users\Chris\pytorch_gpu\Lib\site-packages\ax\service\utils\instantiation.py:258: AxParameterWarning: `sort_values` is not specified for `ChoiceParameter` "architecture". Defaulting to `False` for parameters of `ParameterType` STRING. To override this behavior (or avoid this warning), specify `sort_values` during `ChoiceParameter` construction.
  return ChoiceParameter(
[INFO 02-17 08:43:16] ax.service.utils.instantiation: Inferred value type of ParameterType.FLOAT for parameter dropout_rate. If that is not the


Trial 1/32
Parameters: {'dropout_rate': 0.36588019132614136, 'weight_decay': 0.002128345013887013, 'lr': 3.7883021895735e-05, 'optimizer_type': 'adam', 'batch_size': 64, 'early_stopping_patience': 19, 'architecture': 'expand_2L_xlarge', 'activation': 'leaky_relu', 'normalization': 'layer_norm'}
Using device: cuda
Input dim: 144, Output dim: 804

Fold 0
Train: 29906, Val: 7477, Test: 7599


KeyboardInterrupt: 

In [ ]:
# =====================================================================
# Resume Driving Force NN Bayesian Optimization to 100 trials
# =====================================================================
SAVE_PATH_DF = r"Ax_checkpoints/ax_client_NN_DF(T)_checkpoint.json"

ax_client_df = AxClient.load_from_json_file(SAVE_PATH_DF)

completed_trials = len(ax_client_df.experiment.trials)
target_trials = 100
remaining_trials = target_trials - completed_trials

print(f"Loaded {completed_trials} completed trials from checkpoint")
print(f"Remaining trials to reach {target_trials}: {remaining_trials}")

if remaining_trials > 0:
    for i in range(remaining_trials):
        current_trial = completed_trials + i + 1
        print(f"\n{'='*60}")
        print(f"Trial {current_trial}/{target_trials}")
        print('='*60)

        parameters, trial_index = ax_client_df.get_next_trial()
        print(f"Parameters: {parameters}")

        try:
            result = evaluate_for_ax_df(parameters)
            ax_client_df.complete_trial(trial_index=trial_index, raw_data=result)
            print(f"Result: Non-zero RMSE = {result['avg_rmse_nonzero'][0]:.4f}")
        except Exception as e:
            print(f"Trial failed: {e}")
            ax_client_df.log_trial_failure(trial_index=trial_index)

        # Save checkpoint after each trial
        ax_client_df.save_to_json_file(SAVE_PATH_DF)
        print(f"Checkpoint saved ({current_trial}/{target_trials})")

    best_parameters_df, values_df = ax_client_df.get_best_parameters()
    print(f"\n{'='*60}")
    print("OPTIMIZATION COMPLETE — Driving Force NN (100 trials)")
    print('='*60)
    print(f"Best parameters: {best_parameters_df}")
    print(f"Best Non-zero RMSE: {values_df[0]['avg_rmse_nonzero']:.4f}")
else:
    print("Already at or past 100 trials. No additional trials needed.")
    best_parameters_df, values_df = ax_client_df.get_best_parameters()
    print(f"Best parameters: {best_parameters_df}")
    print(f"Best Non-zero RMSE: {values_df[0]['avg_rmse_nonzero']:.4f}")

c:\Users\Chris\pytorch_gpu\Lib\site-packages\ax\storage\json_store\decoder.py:288: AxParameterWarning: `sort_values` is not specified for `ChoiceParameter` "architecture". Defaulting to `False` for parameters of `ParameterType` STRING. To override this behavior (or avoid this warning), specify `sort_values` during `ChoiceParameter` construction.
  return _class(
c:\Users\Chris\pytorch_gpu\Lib\site-packages\ax\storage\json_store\decoder.py:288: AxParameterWarning: `sort_values` is not specified for `ChoiceParameter` "activation". Defaulting to `False` for parameters of `ParameterType` STRING. To override this behavior (or avoid this warning), specify `sort_values` during `ChoiceParameter` construction.
  return _class(
c:\Users\Chris\pytorch_gpu\Lib\site-packages\ax\storage\json_store\decoder.py:288: AxParameterWarning: `sort_values` is not specified for `ChoiceParameter` "normalization". Defaulting to `False` for parameters of `ParameterType` STRING. To override this behavior (or avoid

Loaded 24 completed trials from checkpoint
Remaining trials to reach 100: 76

Trial 25/100


C:\Users\Chris\python\Lib\dataclasses.py:1581: RuntimeWarning: If using a 2-dim `batch_initial_conditions` botorch will default to old behavior of ignoring `num_restarts` and just use the given `batch_initial_conditions` by setting `raw_samples` to None.
  return obj.__class__(**changes)
C:\Users\Chris\python\Lib\dataclasses.py:1581: RuntimeWarning: If using a 2-dim `batch_initial_conditions` botorch will default to old behavior of ignoring `num_restarts` and just use the given `batch_initial_conditions` by setting `raw_samples` to None.
  return obj.__class__(**changes)
C:\Users\Chris\python\Lib\dataclasses.py:1581: RuntimeWarning: If using a 2-dim `batch_initial_conditions` botorch will default to old behavior of ignoring `num_restarts` and just use the given `batch_initial_conditions` by setting `raw_samples` to None.
  return obj.__class__(**changes)
C:\Users\Chris\python\Lib\dataclasses.py:1581: RuntimeWarning: If using a 2-dim `batch_initial_conditions` botorch will default to ol

Parameters: {'dropout_rate': 0.0, 'weight_decay': 2.2044587686597013e-05, 'lr': 1e-05, 'optimizer_type': 'adam', 'batch_size': 32, 'early_stopping_patience': 44, 'architecture': 'constant_2048', 'activation': 'leaky_relu', 'normalization': 'layer_norm'}
Using device: cuda
Input dim: 144, Output dim: 804

Fold 0
Train: 29906, Val: 7477, Test: 7599
Early stopping at epoch 678
Driving Force (all)      - RMSE: 1.9756, MAE: 0.7127
Driving Force (non-zero) - RMSE: 5.9091, MAE: 3.5707  [498,341 values]

Fold 1
Train: 23174, Val: 5794, Test: 16014
Early stopping at epoch 528
Driving Force (all)      - RMSE: 2.1933, MAE: 0.7093
Driving Force (non-zero) - RMSE: 6.2318, MAE: 3.1930  [1,207,679 values]

Fold 2
Train: 30885, Val: 7722, Test: 6375
Early stopping at epoch 637
Driving Force (all)      - RMSE: 1.4996, MAE: 0.5790
Driving Force (non-zero) - RMSE: 4.0700, MAE: 2.1596  [399,249 values]

Fold 3
Train: 27825, Val: 6957, Test: 10200
Early stopping at epoch 268
Driving Force (all)      - RMSE

[INFO 02-17 11:04:59] ax.service.ax_client: Completed trial 24 with data: {'avg_rmse_nonzero': (np.float32(5.161592), np.float64(0.603488))}.
[INFO 02-17 11:04:59] ax.service.ax_client: Saved JSON-serialized state of optimization to `c:\Users\Chris\OneDrive\Desktop\ML Project\ax_client_DrivingForce(T)_checkpoint.json`.


...

Result: Non-zero RMSE = 5.1454
Checkpoint saved (99/100)

Trial 100/100


C:\Users\Chris\python\Lib\dataclasses.py:1581: RuntimeWarning: If using a 2-dim `batch_initial_conditions` botorch will default to old behavior of ignoring `num_restarts` and just use the given `batch_initial_conditions` by setting `raw_samples` to None.
  return obj.__class__(**changes)
C:\Users\Chris\python\Lib\dataclasses.py:1581: RuntimeWarning: If using a 2-dim `batch_initial_conditions` botorch will default to old behavior of ignoring `num_restarts` and just use the given `batch_initial_conditions` by setting `raw_samples` to None.
  return obj.__class__(**changes)
C:\Users\Chris\python\Lib\dataclasses.py:1581: RuntimeWarning: If using a 2-dim `batch_initial_conditions` botorch will default to old behavior of ignoring `num_restarts` and just use the given `batch_initial_conditions` by setting `raw_samples` to None.
  return obj.__class__(**changes)
C:\Users\Chris\python\Lib\dataclasses.py:1581: RuntimeWarning: If using a 2-dim `batch_initial_conditions` botorch will default to ol

Parameters: {'dropout_rate': 0.0, 'weight_decay': 0.01, 'lr': 1e-05, 'optimizer_type': 'adamw', 'batch_size': 32, 'early_stopping_patience': 50, 'architecture': 'expand_2L_large', 'activation': 'leaky_relu', 'normalization': 'layer_norm'}
Using device: cuda
Input dim: 144, Output dim: 804

Fold 0
Train: 29906, Val: 7477, Test: 7599
Early stopping at epoch 571
Driving Force (all)      - RMSE: 2.1094, MAE: 0.7915
Driving Force (non-zero) - RMSE: 6.3030, MAE: 3.7994  [498,341 values]

Fold 1
Train: 23174, Val: 5794, Test: 16014
Early stopping at epoch 380
Driving Force (all)      - RMSE: 2.1825, MAE: 0.6776
Driving Force (non-zero) - RMSE: 6.5044, MAE: 3.2890  [1,207,679 values]

Fold 2
Train: 30885, Val: 7722, Test: 6375
Early stopping at epoch 560
Driving Force (all)      - RMSE: 1.6864, MAE: 0.6709
Driving Force (non-zero) - RMSE: 4.5672, MAE: 2.4543  [399,249 values]

Fold 3
Train: 27825, Val: 6957, Test: 10200
Early stopping at epoch 296
Driving Force (all)      - RMSE: 2.3357, MAE: 

[INFO 02-19 17:24:23] ax.service.ax_client: Completed trial 99 with data: {'avg_rmse_nonzero': (np.float32(5.3670096), np.float64(0.663292))}.
[INFO 02-19 17:24:23] ax.service.ax_client: Saved JSON-serialized state of optimization to `c:\Users\Chris\OneDrive\Desktop\ML Project\ax_client_DrivingForce(T)_checkpoint.json`.


Result: Non-zero RMSE = 5.3670
Checkpoint saved (100/100)

OPTIMIZATION COMPLETE — Driving Force NN (100 trials)
Best parameters: {'dropout_rate': 0.0, 'weight_decay': 0.01, 'lr': 1e-05, 'optimizer_type': 'adam', 'batch_size': 32, 'early_stopping_patience': 29, 'architecture': 'constant_2048', 'activation': 'leaky_relu', 'normalization': 'layer_norm'}
Best Non-zero RMSE: 5.2925


In [ ]:
ax_client_df = AxClient.load_from_json_file(r"Ax_checkpoints/ax_client_NN_DF(T)_checkpoint.json")
best_parameters_df, values_df = ax_client_df.get_best_parameters()
print(f"\n{'='*60}")
print("OPTIMIZATION COMPLETE — Driving Force NN (100 trials)")
print('='*60)
print(f"Best parameters: {best_parameters_df}")
print(f"Best Non-zero RMSE: {values_df[0]['avg_rmse_nonzero']:.4f}")

c:\Users\Chris\python\Lib\site-packages\ax\storage\json_store\decoder.py:288: AxParameterWarning: `sort_values` is not specified for `ChoiceParameter` "architecture". Defaulting to `False` for parameters of `ParameterType` STRING. To override this behavior (or avoid this warning), specify `sort_values` during `ChoiceParameter` construction.
  return _class(
c:\Users\Chris\python\Lib\site-packages\ax\storage\json_store\decoder.py:288: AxParameterWarning: `sort_values` is not specified for `ChoiceParameter` "activation". Defaulting to `False` for parameters of `ParameterType` STRING. To override this behavior (or avoid this warning), specify `sort_values` during `ChoiceParameter` construction.
  return _class(
c:\Users\Chris\python\Lib\site-packages\ax\storage\json_store\decoder.py:288: AxParameterWarning: `sort_values` is not specified for `ChoiceParameter` "normalization". Defaulting to `False` for parameters of `ParameterType` STRING. To override this behavior (or avoid this warning),


OPTIMIZATION COMPLETE — Driving Force NN (100 trials)
Best parameters: {'dropout_rate': 0.0, 'weight_decay': 0.01, 'lr': 1e-05, 'optimizer_type': 'adam', 'batch_size': 32, 'early_stopping_patience': 29, 'architecture': 'constant_2048', 'activation': 'leaky_relu', 'normalization': 'layer_norm'}
Best Non-zero RMSE: 5.2925


In [ ]:
# =====================================================================
# XGBoost Evaluation & Bayesian Optimization for Phase Fractions (NF)
# =====================================================================

def evaluate_parameters_XGB_NF_FT(parameters, verbose=True):
    """
    Evaluate XGBoost hyperparameters for predicting phase fractions (NF)
    using 5-fold KMeans-based cross-validation (same splits as NN evaluation).

    Parameters:
    -----------
    parameters : dict
        XGBoost hyperparameters including:
        - n_estimators, max_depth, learning_rate, subsample,
          colsample_bytree, min_child_weight, gamma,
          reg_alpha, reg_lambda, early_stopping_rounds
    verbose : bool
        Whether to print progress

    Returns:
    --------
    dict : Results including fold-level and average metrics
    """
    # Extract hyperparameters with defaults
    n_estimators = parameters.get('n_estimators', 500)
    max_depth = parameters.get('max_depth', 6)
    learning_rate = parameters.get('learning_rate', 0.1)
    subsample = parameters.get('subsample', 0.8)
    colsample_bytree = parameters.get('colsample_bytree', 0.8)
    min_child_weight = parameters.get('min_child_weight', 1)
    gamma = parameters.get('gamma', 0.0)
    reg_alpha = parameters.get('reg_alpha', 0.0)
    reg_lambda = parameters.get('reg_lambda', 1.0)
    early_stopping_rounds = parameters.get('early_stopping_rounds', 20)


    # Copy the x and y data — use phase fraction targets
    y_data = y_nf.copy()
    x_data = X_combined.copy()

    # Get dimensions
    input_dim = x_data.shape[1]
    output_dim = y_data.shape[1]
    if verbose:
        print(f"Input dim: {input_dim}, Output dim: {output_dim}")

    # Store results for each fold
    fold_results = []
    all_test_predictions = []
    all_test_targets = []

    for test_group in range(5):
        if verbose:
            print(f"\n{'='*50}")
            print(f"Fold {test_group}")
            print('='*50)

        # Get the split indexes for training and testing
        train_idx = cv_group_df['cv_group'] != test_group
        test_idx = cv_group_df['cv_group'] == test_group

        # Split the x and y data into train and test
        x_train = x_data.loc[train_idx]
        x_test = x_data.loc[test_idx]
        y_train = y_data.loc[train_idx]
        y_test = y_data.loc[test_idx]

        # Create train and validation split
        X_train, X_val, y_train_split, y_val = train_test_split(
            x_train, y_train, test_size=0.2, random_state=42
        )

        if verbose:
            print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(x_test)}")

        # Train one XGBRegressor per output column (manual multi-output)
        # This ensures proper GPU usage — native multi-output doesn't fully support GPU
        estimators = []
        best_iters = []
        output_cols = y_train_split.columns.tolist()

        for col_idx, col_name in enumerate(output_cols):
            xgb_model = XGBRegressor(
                n_estimators=n_estimators,
                max_depth=max_depth,
                learning_rate=learning_rate,
                subsample=subsample,
                colsample_bytree=colsample_bytree,
                min_child_weight=min_child_weight,
                gamma=gamma,
                reg_alpha=reg_alpha,
                reg_lambda=reg_lambda,
                tree_method='hist',
                device='cuda',
                random_state=42,
                n_jobs=-1,
                verbosity=0,
                early_stopping_rounds=early_stopping_rounds,
                eval_metric='rmse',
            )

            xgb_model.fit(
                X_train, y_train_split[col_name],
                eval_set=[(X_val, y_val[col_name])],
                verbose=False,
            )

            estimators.append(xgb_model)
            best_iters.append(xgb_model.best_iteration)

        if verbose:
            print(f"Best iterations (min/mean/max): {min(best_iters)}/{np.mean(best_iters):.0f}/{max(best_iters)}")

        # Predict on test set (stack individual predictions)
        test_preds = np.column_stack([
            est.predict(x_test) for est in estimators
        ])
        test_targets = y_test.values

        # Get column names for NF columns
        col_names = y_data.columns.tolist()
        nf_col_idxs = [i for i, col in enumerate(col_names) if col.startswith('NF_')]

        # Calculate overall RMSE and MAE (all values)
        mse_original = np.mean((test_preds - test_targets) ** 2)
        rmse = np.sqrt(mse_original)
        mae = np.mean(np.abs(test_preds - test_targets))

        # Calculate metrics for non-zero targets only
        nonzero_threshold = 1e-6

        nonzero_mask = np.abs(test_targets) > nonzero_threshold
        if np.any(nonzero_mask):
            nonzero_errors = test_preds[nonzero_mask] - test_targets[nonzero_mask]
            rmse_nonzero = np.sqrt(np.mean(nonzero_errors ** 2))
            mae_nonzero = np.mean(np.abs(nonzero_errors))
            n_nonzero = np.sum(nonzero_mask)
        else:
            rmse_nonzero, mae_nonzero, n_nonzero = 0.0, 0.0, 0

        # Phase Fraction (NF) specific metrics — non-zero only
        if nf_col_idxs:
            nf_preds = test_preds[:, nf_col_idxs]
            nf_targets = test_targets[:, nf_col_idxs]
            nf_nonzero_mask = np.abs(nf_targets) > nonzero_threshold
            if np.any(nf_nonzero_mask):
                nf_errors = nf_preds[nf_nonzero_mask] - nf_targets[nf_nonzero_mask]
                nf_rmse = np.sqrt(np.mean(nf_errors ** 2))
                nf_mae = np.mean(np.abs(nf_errors))
                n_nf_nonzero = np.sum(nf_nonzero_mask)
            else:
                nf_rmse, nf_mae, n_nf_nonzero = 0.0, 0.0, 0
        else:
            nf_rmse, nf_mae, n_nf_nonzero = 0.0, 0.0, 0

        if verbose:
            print(f"Overall (all)      - RMSE: {rmse:.4f}, MAE: {mae:.4f}")
            print(f"Overall (non-zero) - RMSE: {rmse_nonzero:.4f}, MAE: {mae_nonzero:.4f}  [{n_nonzero:,} values]")
            print(f"Phase Fraction (NF) - RMSE: {nf_rmse:.4f}, MAE: {nf_mae:.4f}  [{n_nf_nonzero:,} non-zero values]")

        # Store fold results
        fold_results.append({
            'fold': test_group,
            'test_loss': mse_original,
            'test_rmse': rmse,
            'test_mae': mae,
            'rmse_nonzero': rmse_nonzero,
            'mae_nonzero': mae_nonzero,
            'nf_rmse': nf_rmse,
            'nf_mae': nf_mae,
        })

        all_test_predictions.append(test_preds)
        all_test_targets.append(test_targets)

    # Summary across all folds
    if verbose:
        print(f"\n{'='*50}")
        print("Cross-Validation Summary — XGBoost Phase Fractions (NF)")
        print('='*50)

    avg_test_loss = np.mean([r['test_loss'] for r in fold_results])
    avg_test_rmse = np.mean([r['test_rmse'] for r in fold_results])
    avg_test_mae = np.mean([r['test_mae'] for r in fold_results])
    avg_rmse_nonzero = np.mean([r['rmse_nonzero'] for r in fold_results])
    avg_mae_nonzero = np.mean([r['mae_nonzero'] for r in fold_results])
    avg_nf_rmse = np.mean([r['nf_rmse'] for r in fold_results])
    avg_nf_mae = np.mean([r['nf_mae'] for r in fold_results])
    std_test_loss = np.std([r['test_loss'] for r in fold_results])

    if verbose:
        print(f"Overall (all values):")
        print(f"  Avg Test MSE: {avg_test_loss:.4f} (+/- {std_test_loss:.4f})")
        print(f"  Avg RMSE: {avg_test_rmse:.4f}, MAE: {avg_test_mae:.4f}")
        print(f"\nOverall (non-zero only):")
        print(f"  Avg RMSE: {avg_rmse_nonzero:.4f}, MAE: {avg_mae_nonzero:.4f}")
        print(f"\nPhase Fraction (NF, non-zero only):")
        print(f"  Avg RMSE: {avg_nf_rmse:.4f}, MAE: {avg_nf_mae:.4f}")

    return {
        'fold_results': fold_results,
        'avg_test_loss': avg_test_loss,
        'avg_test_rmse': avg_test_rmse,
        'avg_test_mae': avg_test_mae,
        'avg_rmse_nonzero': avg_rmse_nonzero,
        'avg_mae_nonzero': avg_mae_nonzero,
        'avg_nf_rmse': avg_nf_rmse,
        'avg_nf_mae': avg_nf_mae,
        'all_predictions': all_test_predictions,
        'all_targets': all_test_targets,
    }


# =====================================================================
# Ax Bayesian Optimization for XGBoost NF f(T)
# =====================================================================

ax_client_xgb_nf = AxClient()
ax_client_xgb_nf.create_experiment(
    name="XGB opt NF f(T) Calphed",
    parameters=[
        {
            "name": "n_estimators",
            "type": "range",
            "bounds": [100, 5000],
            "value_type": "int",
        },
        {
            "name": "max_depth",
            "type": "range",
            "bounds": [3, 100],
            "value_type": "int",
        },
        {
            "name": "learning_rate",
            "type": "range",
            "bounds": [0.005, 0.3],
            "log_scale": True,
        },
        {
            "name": "subsample",
            "type": "range",
            "bounds": [0.5, 1.0],
        },
        {
            "name": "colsample_bytree",
            "type": "range",
            "bounds": [0.3, 1.0],
        },
        {
            "name": "min_child_weight",
            "type": "range",
            "bounds": [1, 20],
            "value_type": "int",
        },
        {
            "name": "gamma",
            "type": "range",
            "bounds": [0.0, 5.0],
        },
        {
            "name": "reg_alpha",
            "type": "range",
            "bounds": [1e-6, 10.0],
            "log_scale": True,
        },
        {
            "name": "reg_lambda",
            "type": "range",
            "bounds": [1e-6, 10.0],
            "log_scale": True,
        },
        {
            "name": "early_stopping_rounds",
            "type": "range",
            "bounds": [10, 50],
            "value_type": "int",
        },
    ],
    objectives={"avg_rmse_nonzero": ObjectiveProperties(minimize=True)},
)


def evaluate_for_ax_xgb_nf(parameterization):
    """Wrapper to convert Ax parameters to evaluate_parameters_XGB_NF_FT format."""
    parameters = {
        "n_estimators": parameterization["n_estimators"],
        "max_depth": parameterization["max_depth"],
        "learning_rate": parameterization["learning_rate"],
        "subsample": parameterization["subsample"],
        "colsample_bytree": parameterization["colsample_bytree"],
        "min_child_weight": parameterization["min_child_weight"],
        "gamma": parameterization["gamma"],
        "reg_alpha": parameterization["reg_alpha"],
        "reg_lambda": parameterization["reg_lambda"],
        "early_stopping_rounds": parameterization["early_stopping_rounds"],
    }

    results = evaluate_parameters_XGB_NF_FT(parameters, verbose=False)

    # Calculate SEM from fold results
    rmse_nonzero_values = [r["rmse_nonzero"] for r in results["fold_results"]]
    mean_rmse_nonzero = np.mean(rmse_nonzero_values)
    std_rmse_nonzero = np.std(rmse_nonzero_values, ddof=1)
    sem_rmse_nonzero = std_rmse_nonzero / np.sqrt(len(rmse_nonzero_values))

    return {"avg_rmse_nonzero": (mean_rmse_nonzero, sem_rmse_nonzero)}


# =====================================================================
# Run the Bayesian Optimization Loop — XGBoost NF f(T)
# =====================================================================
n_trials_xgb_nf = 100
SAVE_PATH_XGB_NF = r"Ax_checkpoints/ax_client_XGB_NF(T)_checkpoint.json"

for i in range(n_trials_xgb_nf):
    print(f"\n{'='*60}")
    print(f"Trial {i+1}/{n_trials_xgb_nf}")
    print('='*60)

    parameters, trial_index = ax_client_xgb_nf.get_next_trial()
    print(f"Parameters: {parameters}")

    try:
        result = evaluate_for_ax_xgb_nf(parameters)
        ax_client_xgb_nf.complete_trial(trial_index=trial_index, raw_data=result)
        print(f"Result: Non-zero RMSE = {result['avg_rmse_nonzero'][0]:.4f}")
    except Exception as e:
        print(f"Trial failed: {e}")
        ax_client_xgb_nf.log_trial_failure(trial_index=trial_index)

    # Save checkpoint after each trial
    ax_client_xgb_nf.save_to_json_file(SAVE_PATH_XGB_NF)
    print(f"Checkpoint saved to {SAVE_PATH_XGB_NF}")

# Get best parameters
best_parameters_xgb_nf, values_xgb_nf = ax_client_xgb_nf.get_best_parameters()
print(f"\n{'='*60}")
print("OPTIMIZATION COMPLETE — XGBoost Phase Fractions NF f(T)")
print('='*60)
print(f"Best parameters: {best_parameters_xgb_nf}")
print(f"Best Non-zero RMSE: {values_xgb_nf[0]['avg_rmse_nonzero']:.4f}")

[INFO 02-25 17:35:16] ax.service.ax_client: Starting optimization with verbose logging. To disable logging, set the `verbose_logging` argument to `False`. Note that float values in the logs are rounded to 6 decimal points.
[INFO 02-25 17:35:16] ax.service.utils.instantiation: Inferred value type of ParameterType.FLOAT for parameter learning_rate. If that is not the expected value type, you can explicitly specify 'value_type' ('int', 'float', 'bool' or 'str') in parameter dict.
[INFO 02-25 17:35:16] ax.service.utils.instantiation: Inferred value type of ParameterType.FLOAT for parameter subsample. If that is not the expected value type, you can explicitly specify 'value_type' ('int', 'float', 'bool' or 'str') in parameter dict.
[INFO 02-25 17:35:16] ax.service.utils.instantiation: Inferred value type of ParameterType.FLOAT for parameter colsample_bytree. If that is not the expected value type, you can explicitly specify 'value_type' ('int', 'float', 'bool' or 'str') in parameter dict.


[INFO 02-25 17:35:16] ax.service.utils.instantiation: Inferred value type of ParameterType.FLOAT for parameter gamma. If that is not the expected value type, you can explicitly specify 'value_type' ('int', 'float', 'bool' or 'str') in parameter dict.
[INFO 02-25 17:35:16] ax.service.utils.instantiation: Inferred value type of ParameterType.FLOAT for parameter reg_alpha. If that is not the expected value type, you can explicitly specify 'value_type' ('int', 'float', 'bool' or 'str') in parameter dict.
[INFO 02-25 17:35:16] ax.service.utils.instantiation: Inferred value type of ParameterType.FLOAT for parameter reg_lambda. If that is not the expected value type, you can explicitly specify 'value_type' ('int', 'float', 'bool' or 'str') in parameter dict.
[INFO 02-25 17:35:16] ax.service.utils.instantiation: Created search space: SearchSpace(parameters=[RangeParameter(name='n_estimators', parameter_type=INT, range=[100, 5000]), RangeParameter(name='max_depth', parameter_type=INT, range=[3,


Trial 1/100
Parameters: {'n_estimators': 846, 'max_depth': 61, 'learning_rate': 0.03719206554248498, 'subsample': 0.7541288733482361, 'colsample_bytree': 0.5477683752775192, 'min_child_weight': 20, 'gamma': 0.4652963951230049, 'reg_alpha': 5.635551059708466, 'reg_lambda': 6.097468206357738e-05, 'early_stopping_rounds': 29}


[INFO 02-25 17:58:59] ax.service.ax_client: Completed trial 0 with data: {'avg_rmse_nonzero': (0.37052, 0.029078)}.
[INFO 02-25 17:58:59] ax.service.ax_client: Saved JSON-serialized state of optimization to `c:\Users\Chris\OneDrive\Desktop\ML Project\ax_client_XGB_NF_fT_checkpoint.json`.
c:\Users\Chris\python\Lib\site-packages\ax\modelbridge\cross_validation.py:439: UserWarning: Encountered exception in computing model fit quality: RandomModelBridge does not support prediction.
  warn("Encountered exception in computing model fit quality: " + str(e))
[INFO 02-25 17:58:59] ax.service.ax_client: Generated new trial 1 with parameters {'n_estimators': 4041, 'max_depth': 30, 'learning_rate': 0.042852, 'subsample': 0.660767, 'colsample_bytree': 0.734029, 'min_child_weight': 7, 'gamma': 4.930101, 'reg_alpha': 0.00025, 'reg_lambda': 0.818885, 'early_stopping_rounds': 49} using model Sobol.


Result: Non-zero RMSE = 0.3705
Checkpoint saved to c:\Users\Chris\OneDrive\Desktop\ML Project\ax_client_XGB_NF_fT_checkpoint.json

Trial 2/100
Parameters: {'n_estimators': 4041, 'max_depth': 30, 'learning_rate': 0.04285210387136406, 'subsample': 0.6607672781683505, 'colsample_bytree': 0.734029318485409, 'min_child_weight': 7, 'gamma': 4.9301014468073845, 'reg_alpha': 0.00024954599759043436, 'reg_lambda': 0.8188851431198646, 'early_stopping_rounds': 49}


...

Result: Non-zero RMSE = 0.3788
Checkpoint saved to c:\Users\Chris\OneDrive\Desktop\ML Project\ax_client_XGB_NF_fT_checkpoint.json

Trial 100/100


c:\Users\Chris\python\Lib\dataclasses.py:1581: RuntimeWarning: If using a 2-dim `batch_initial_conditions` botorch will default to old behavior of ignoring `num_restarts` and just use the given `batch_initial_conditions` by setting `raw_samples` to None.
  return obj.__class__(**changes)
c:\Users\Chris\python\Lib\dataclasses.py:1581: RuntimeWarning: If using a 2-dim `batch_initial_conditions` botorch will default to old behavior of ignoring `num_restarts` and just use the given `batch_initial_conditions` by setting `raw_samples` to None.
  return obj.__class__(**changes)
c:\Users\Chris\python\Lib\dataclasses.py:1581: RuntimeWarning: If using a 2-dim `batch_initial_conditions` botorch will default to old behavior of ignoring `num_restarts` and just use the given `batch_initial_conditions` by setting `raw_samples` to None.
  return obj.__class__(**changes)
c:\Users\Chris\python\Lib\dataclasses.py:1581: RuntimeWarning: If using a 2-dim `batch_initial_conditions` botorch will default to ol

Parameters: {'n_estimators': 4056, 'max_depth': 21, 'learning_rate': 0.016676076252937135, 'subsample': 1.0, 'colsample_bytree': 1.0, 'min_child_weight': 8, 'gamma': 0.0, 'reg_alpha': 2.5739854966974115e-05, 'reg_lambda': 1e-06, 'early_stopping_rounds': 10}


[INFO 03-01 01:20:02] ax.service.ax_client: Completed trial 99 with data: {'avg_rmse_nonzero': (0.397241, 0.041343)}.
[INFO 03-01 01:20:02] ax.service.ax_client: Saved JSON-serialized state of optimization to `c:\Users\Chris\OneDrive\Desktop\ML Project\ax_client_XGB_NF_fT_checkpoint.json`.


Result: Non-zero RMSE = 0.3972
Checkpoint saved to c:\Users\Chris\OneDrive\Desktop\ML Project\ax_client_XGB_NF_fT_checkpoint.json

OPTIMIZATION COMPLETE — XGBoost Phase Fractions NF f(T)
Best parameters: {'n_estimators': 4529, 'max_depth': 85, 'learning_rate': 0.005, 'subsample': 0.5, 'colsample_bytree': 0.3, 'min_child_weight': 16, 'gamma': 0.0, 'reg_alpha': 2.2856341343345186, 'reg_lambda': 0.66622229114204, 'early_stopping_rounds': 50}
Best Non-zero RMSE: 0.3643


In [ ]:
ax_client_xgb_nf = AxClient.load_from_json_file(r"Ax_checkpoints/ax_client_XGB_NF(T)_checkpoint.json")
best_parameters_xgb_nf, values_xgb_nf = ax_client_xgb_nf.get_best_parameters()
print(f"\n{'='*60}")
print("OPTIMIZATION COMPLETE — XGBoost Phase Fractions NF f(T)")
print('='*60)
print(f"Best parameters: {best_parameters_xgb_nf}")
print(f"Best Non-zero RMSE: {values_xgb_nf[0]['avg_rmse_nonzero']:.4f}")

[INFO 03-06 11:32:37] ax.service.ax_client: Starting optimization with verbose logging. To disable logging, set the `verbose_logging` argument to `False`. Note that float values in the logs are rounded to 6 decimal points.


KeyboardInterrupt: 

In [ ]:
# =====================================================================
# XGBoost Evaluation & Bayesian Optimization for Driving Force XGB (DF)
# =====================================================================

def evaluate_parameters_XGB_DF_FT(parameters, verbose=True):
    """
    Evaluate XGBoost hyperparameters for predicting driving force (DF)
    using 5-fold KMeans-based cross-validation (same splits as NN evaluation).

    Parameters:
    -----------
    parameters : dict
        XGBoost hyperparameters including:
        - n_estimators, max_depth, learning_rate, subsample,
          colsample_bytree, min_child_weight, gamma,
          reg_alpha, reg_lambda, early_stopping_rounds
    verbose : bool
        Whether to print progress

    Returns:
    --------
    dict : Results including fold-level and average metrics
    """
    # Extract hyperparameters with defaults
    n_estimators = parameters.get('n_estimators', 500)
    max_depth = parameters.get('max_depth', 6)
    learning_rate = parameters.get('learning_rate', 0.1)
    subsample = parameters.get('subsample', 0.8)
    colsample_bytree = parameters.get('colsample_bytree', 0.8)
    min_child_weight = parameters.get('min_child_weight', 1)
    gamma = parameters.get('gamma', 0.0)
    reg_alpha = parameters.get('reg_alpha', 0.0)
    reg_lambda = parameters.get('reg_lambda', 1.0)
    early_stopping_rounds = parameters.get('early_stopping_rounds', 20)


    # Copy the x and y data — use driving force targets
    y_data = y_df.copy()
    x_data = X_combined.copy()

    # Get dimensions
    input_dim = x_data.shape[1]
    output_dim = y_data.shape[1]
    if verbose:
        print(f"Input dim: {input_dim}, Output dim: {output_dim}")

    # Store results for each fold
    fold_results = []
    all_test_predictions = []
    all_test_targets = []

    for test_group in range(5):
        if verbose:
            print(f"\n{'='*50}")
            print(f"Fold {test_group}")
            print('='*50)

        # Get the split indexes for training and testing
        train_idx = cv_group_df['cv_group'] != test_group
        test_idx = cv_group_df['cv_group'] == test_group

        # Split the x and y data into train and test
        x_train = x_data.loc[train_idx]
        x_test = x_data.loc[test_idx]
        y_train = y_data.loc[train_idx]
        y_test = y_data.loc[test_idx]

        # Create train and validation split
        X_train, X_val, y_train_split, y_val = train_test_split(
            x_train, y_train, test_size=0.2, random_state=42
        )

        if verbose:
            print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(x_test)}")

        # Train one XGBRegressor per output column (manual multi-output)
        # This ensures proper GPU usage — native multi-output doesn't fully support GPU
        estimators = []
        best_iters = []
        output_cols = y_train_split.columns.tolist()

        for col_idx, col_name in enumerate(output_cols):
            xgb_model = XGBRegressor(
                n_estimators=n_estimators,
                max_depth=max_depth,
                learning_rate=learning_rate,
                subsample=subsample,
                colsample_bytree=colsample_bytree,
                min_child_weight=min_child_weight,
                gamma=gamma,
                reg_alpha=reg_alpha,
                reg_lambda=reg_lambda,
                tree_method='hist',
                device='cuda',
                random_state=42,
                n_jobs=-1,
                verbosity=0,
                early_stopping_rounds=early_stopping_rounds,
                eval_metric='rmse',
            )

            xgb_model.fit(
                X_train, y_train_split[col_name],
                eval_set=[(X_val, y_val[col_name])],
                verbose=False,
            )

            estimators.append(xgb_model)
            best_iters.append(xgb_model.best_iteration)

        if verbose:
            print(f"Best iterations (min/mean/max): {min(best_iters)}/{np.mean(best_iters):.0f}/{max(best_iters)}")

        # Predict on test set (stack individual predictions)
        test_preds = np.column_stack([
            est.predict(x_test) for est in estimators
        ])
        test_targets = y_test.values

        # Get column names for DF columns
        col_names = y_data.columns.tolist()
        df_col_idxs = [i for i, col in enumerate(col_names) if col.startswith('DF_')]

        # Calculate overall RMSE and MAE (all values)
        mse_original = np.mean((test_preds - test_targets) ** 2)
        rmse = np.sqrt(mse_original)
        mae = np.mean(np.abs(test_preds - test_targets))

        # Calculate metrics for non-zero targets only
        nonzero_threshold = 1e-6

        nonzero_mask = np.abs(test_targets) > nonzero_threshold
        if np.any(nonzero_mask):
            nonzero_errors = test_preds[nonzero_mask] - test_targets[nonzero_mask]
            rmse_nonzero = np.sqrt(np.mean(nonzero_errors ** 2))
            mae_nonzero = np.mean(np.abs(nonzero_errors))
            n_nonzero = np.sum(nonzero_mask)
        else:
            rmse_nonzero, mae_nonzero, n_nonzero = 0.0, 0.0, 0

        # Driving Force (DF) specific metrics — non-zero only
        if df_col_idxs:
            df_preds = test_preds[:, df_col_idxs]
            df_targets = test_targets[:, df_col_idxs]
            df_nonzero_mask = np.abs(df_targets) > nonzero_threshold
            if np.any(df_nonzero_mask):
                df_errors = df_preds[df_nonzero_mask] - df_targets[df_nonzero_mask]
                nf_rmse = np.sqrt(np.mean(df_errors ** 2))
                nf_mae = np.mean(np.abs(df_errors))
                n_nf_nonzero = np.sum(df_nonzero_mask)
            else:
                nf_rmse, nf_mae, n_nf_nonzero = 0.0, 0.0, 0
        else:
            nf_rmse, nf_mae, n_nf_nonzero = 0.0, 0.0, 0

        if verbose:
            print(f"Overall (all)      - RMSE: {rmse:.4f}, MAE: {mae:.4f}")
            print(f"Overall (non-zero) - RMSE: {rmse_nonzero:.4f}, MAE: {mae_nonzero:.4f}  [{n_nonzero:,} values]")
            print(f"Phase Fraction (NF) - RMSE: {nf_rmse:.4f}, MAE: {nf_mae:.4f}  [{n_nf_nonzero:,} non-zero values]")

        # Store fold results
        fold_results.append({
            'fold': test_group,
            'test_loss': mse_original,
            'test_rmse': rmse,
            'test_mae': mae,
            'rmse_nonzero': rmse_nonzero,
            'mae_nonzero': mae_nonzero,
            'nf_rmse': nf_rmse,
            'nf_mae': nf_mae,
        })

        all_test_predictions.append(test_preds)
        all_test_targets.append(test_targets)

    # Summary across all folds
    if verbose:
        print(f"\n{'='*50}")
        print("Cross-Validation Summary — XGBoost Phase Fractions (NF)")
        print('='*50)

    avg_test_loss = np.mean([r['test_loss'] for r in fold_results])
    avg_test_rmse = np.mean([r['test_rmse'] for r in fold_results])
    avg_test_mae = np.mean([r['test_mae'] for r in fold_results])
    avg_rmse_nonzero = np.mean([r['rmse_nonzero'] for r in fold_results])
    avg_mae_nonzero = np.mean([r['mae_nonzero'] for r in fold_results])
    avg_nf_rmse = np.mean([r['nf_rmse'] for r in fold_results])
    avg_nf_mae = np.mean([r['nf_mae'] for r in fold_results])
    std_test_loss = np.std([r['test_loss'] for r in fold_results])

    if verbose:
        print(f"Overall (all values):")
        print(f"  Avg Test MSE: {avg_test_loss:.4f} (+/- {std_test_loss:.4f})")
        print(f"  Avg RMSE: {avg_test_rmse:.4f}, MAE: {avg_test_mae:.4f}")
        print(f"\nOverall (non-zero only):")
        print(f"  Avg RMSE: {avg_rmse_nonzero:.4f}, MAE: {avg_mae_nonzero:.4f}")
        print(f"\nPhase Fraction (NF, non-zero only):")
        print(f"  Avg RMSE: {avg_nf_rmse:.4f}, MAE: {avg_nf_mae:.4f}")

    return {
        'fold_results': fold_results,
        'avg_test_loss': avg_test_loss,
        'avg_test_rmse': avg_test_rmse,
        'avg_test_mae': avg_test_mae,
        'avg_rmse_nonzero': avg_rmse_nonzero,
        'avg_mae_nonzero': avg_mae_nonzero,
        'avg_nf_rmse': avg_nf_rmse,
        'avg_nf_mae': avg_nf_mae,
        'all_predictions': all_test_predictions,
        'all_targets': all_test_targets,
    }


# =====================================================================
# Ax Bayesian Optimization for XGBoost DF f(T)
# =====================================================================

ax_client_xgb_df = AxClient()
ax_client_xgb_df.create_experiment(
    name="XGB opt DF f(T) Calphed",
    parameters=[
        {
            "name": "n_estimators",
            "type": "range",
            "bounds": [100, 5000],
            "value_type": "int",
        },
        {
            "name": "max_depth",
            "type": "range",
            "bounds": [3, 100],
            "value_type": "int",
        },
        {
            "name": "learning_rate",
            "type": "range",
            "bounds": [0.005, 0.3],
            "log_scale": True,
        },
        {
            "name": "subsample",
            "type": "range",
            "bounds": [0.5, 1.0],
        },
        {
            "name": "colsample_bytree",
            "type": "range",
            "bounds": [0.3, 1.0],
        },
        {
            "name": "min_child_weight",
            "type": "range",
            "bounds": [1, 20],
            "value_type": "int",
        },
        {
            "name": "gamma",
            "type": "range",
            "bounds": [0.0, 5.0],
        },
        {
            "name": "reg_alpha",
            "type": "range",
            "bounds": [1e-6, 10.0],
            "log_scale": True,
        },
        {
            "name": "reg_lambda",
            "type": "range",
            "bounds": [1e-6, 10.0],
            "log_scale": True,
        },
        {
            "name": "early_stopping_rounds",
            "type": "range",
            "bounds": [10, 50],
            "value_type": "int",
        },
    ],
    objectives={"avg_rmse_nonzero": ObjectiveProperties(minimize=True)},
)


def evaluate_for_ax_xgb_df(parameterization):
    """Wrapper to convert Ax parameters to evaluate_parameters_XGB_DF_FT format."""
    parameters = {
        "n_estimators": parameterization["n_estimators"],
        "max_depth": parameterization["max_depth"],
        "learning_rate": parameterization["learning_rate"],
        "subsample": parameterization["subsample"],
        "colsample_bytree": parameterization["colsample_bytree"],
        "min_child_weight": parameterization["min_child_weight"],
        "gamma": parameterization["gamma"],
        "reg_alpha": parameterization["reg_alpha"],
        "reg_lambda": parameterization["reg_lambda"],
        "early_stopping_rounds": parameterization["early_stopping_rounds"],
    }

    results = evaluate_parameters_XGB_DF_FT(parameters, verbose=False)

    # Calculate SEM from fold results
    rmse_nonzero_values = [r["rmse_nonzero"] for r in results["fold_results"]]
    mean_rmse_nonzero = np.mean(rmse_nonzero_values)
    std_rmse_nonzero = np.std(rmse_nonzero_values, ddof=1)
    sem_rmse_nonzero = std_rmse_nonzero / np.sqrt(len(rmse_nonzero_values))

    return {"avg_rmse_nonzero": (mean_rmse_nonzero, sem_rmse_nonzero)}

SAVE_PATH_XGB_DF = r"Ax_checkpoints/ax_client_XGB_DF(T)_checkpoint.json"


[INFO 03-11 06:06:42] ax.service.ax_client: Starting optimization with verbose logging. To disable logging, set the `verbose_logging` argument to `False`. Note that float values in the logs are rounded to 6 decimal points.
[INFO 03-11 06:06:42] ax.service.utils.instantiation: Inferred value type of ParameterType.FLOAT for parameter learning_rate. If that is not the expected value type, you can explicitly specify 'value_type' ('int', 'float', 'bool' or 'str') in parameter dict.
[INFO 03-11 06:06:42] ax.service.utils.instantiation: Inferred value type of ParameterType.FLOAT for parameter subsample. If that is not the expected value type, you can explicitly specify 'value_type' ('int', 'float', 'bool' or 'str') in parameter dict.
[INFO 03-11 06:06:42] ax.service.utils.instantiation: Inferred value type of ParameterType.FLOAT for parameter colsample_bytree. If that is not the expected value type, you can explicitly specify 'value_type' ('int', 'float', 'bool' or 'str') in parameter dict.
[

In [ ]:
# =====================================================================
# Run the Bayesian Optimization Loop — XGBoost DF f(T)
# =====================================================================
n_trials_xgb_df = 100
SAVE_PATH_XGB_DF = r"Ax_checkpoints/ax_client_XGB_DF(T)_checkpoint.json"


for i in range(n_trials_xgb_df):
    print(f"\n{'='*60}")
    print(f"Trial {i+1}/{n_trials_xgb_df}")
    print('='*60)

    parameters, trial_index = ax_client_xgb_df.get_next_trial()
    print(f"Parameters: {parameters}")

    try:
        result = evaluate_for_ax_xgb_df(parameters)
        ax_client_xgb_df.complete_trial(trial_index=trial_index, raw_data=result)
        print(f"Result: Non-zero RMSE = {result['avg_rmse_nonzero'][0]:.4f}")
    except Exception as e:
        print(f"Trial failed: {e}")
        ax_client_xgb_df.log_trial_failure(trial_index=trial_index)
    # Save checkpoint after each trial
    ax_client_xgb_df.save_to_json_file(SAVE_PATH_XGB_DF)
    print(f"Checkpoint saved to {SAVE_PATH_XGB_DF}")

# Get best parameters
best_parameters_xgb_df, values_xgb_df = ax_client_xgb_df.get_best_parameters()
print(f"\n{'='*60}")
print("OPTIMIZATION COMPLETE — XGBoost Phase Fractions DF f(T)")
print('='*60)
print(f"Best parameters: {best_parameters_xgb_df}")
print(f"Best Non-zero RMSE: {values_xgb_df[0]['avg_rmse_nonzero']:.4f}")

In [13]:
# Load the XGBoost Ax client from last save and continue optimization to 100 trials
ax_client_xgb_df = AxClient.load_from_json_file(SAVE_PATH_XGB_DF)

completed_trials = len(ax_client_xgb_df.experiment.trials)
target_trials = 100
remaining_trials = target_trials - completed_trials

print(f"Loaded {completed_trials} completed trials from {SAVE_PATH_XGB_DF}")
print(f"Remaining trials to reach {target_trials}: {remaining_trials}")

if remaining_trials > 0:
    for i in range(remaining_trials):
        current_trial = completed_trials + i + 1
        print(f"\n{'='*60}")
        print(f"Trial {current_trial}/{target_trials}")
        print('='*60)

        parameters, trial_index = ax_client_xgb_df.get_next_trial()
        print(f"Parameters: {parameters}")

        try:
            result = evaluate_for_ax_xgb_df(parameters)
            ax_client_xgb_df.complete_trial(trial_index=trial_index, raw_data=result)
            print(f"Result: Non-zero RMSE = {result['avg_rmse_nonzero'][0]:.4f}")
        except Exception as e:
            print(f"Trial failed: {e}")
            ax_client_xgb_df.log_trial_failure(trial_index=trial_index)

        # Save after each trial for safety
        ax_client_xgb_df.save_to_json_file(SAVE_PATH_XGB_DF)

    # Get best parameters after all trials
    best_parameters, values = ax_client_xgb_df.get_best_parameters()
    print(f"\n{'='*60}")
    print("OPTIMIZATION COMPLETE")
    print('='*60)
    print(f"Best parameters: {best_parameters}")
    print(f"Best Non-zero RMSE: {values[0]['avg_rmse_nonzero']:.4f}")
else:
    print("Already at or past 100 trials. No additional trials needed.")
    best_parameters, values = ax_client_xgb_df.get_best_parameters()
    print(f"Best parameters: {best_parameters}")
    print(f"Best Non-zero RMSE: {values[0]['avg_rmse_nonzero']:.4f}")

[INFO 03-11 06:06:59] ax.service.ax_client: Starting optimization with verbose logging. To disable logging, set the `verbose_logging` argument to `False`. Note that float values in the logs are rounded to 6 decimal points.
c:\Users\Chris\pytorch_gpu\Lib\site-packages\ax\core\data.py:295: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return cls(df=pd.concat(dfs, axis=0, sort=True))
c:\Users\Chris\pytorch_gpu\Lib\site-packages\ax\modelbridge\cross_validation.py:439: UserWarning: Encountered exception in computing model fit quality: RandomModelBridge does not support prediction.
  warn("Encountered exception in computing model fit quality: " + str(e))
[INFO 03-11 06:06:59] ax.service.ax_client: Generated new trial 83 with parameters {'n_estima

Loaded 83 completed trials from c:\Users\Chris\OneDrive\Desktop\ML Project\ax_client_XGB_DF_fT_checkpoint.json
Remaining trials to reach 100: 17

Trial 84/100
Parameters: {'n_estimators': 3243, 'max_depth': 79, 'learning_rate': 0.03789960863323415, 'subsample': 0.6174330250360072, 'colsample_bytree': 0.8457865629345178, 'min_child_weight': 8, 'gamma': 0.2539283502846956, 'reg_alpha': 0.5948690377718296, 'reg_lambda': 1.2901651382278928e-05, 'early_stopping_rounds': 46}


[INFO 03-11 08:49:01] ax.service.ax_client: Completed trial 83 with data: {'avg_rmse_nonzero': (np.float64(4.476533), np.float64(0.776915))}.
[INFO 03-11 08:49:01] ax.service.ax_client: Saved JSON-serialized state of optimization to `c:\Users\Chris\OneDrive\Desktop\ML Project\ax_client_XGB_DF_fT_checkpoint.json`.
c:\Users\Chris\pytorch_gpu\Lib\site-packages\ax\core\data.py:295: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return cls(df=pd.concat(dfs, axis=0, sort=True))
c:\Users\Chris\pytorch_gpu\Lib\site-packages\ax\modelbridge\cross_validation.py:439: UserWarning: Encountered exception in computing model fit quality: RandomModelBridge does not support prediction.
  warn("Encountered exception in computing model fit quality: " + str(e))
[

Result: Non-zero RMSE = 4.4765

Trial 85/100
Parameters: {'n_estimators': 3993, 'max_depth': 19, 'learning_rate': 0.25125954230322783, 'subsample': 0.848032443318516, 'colsample_bytree': 0.6962485030293464, 'min_child_weight': 4, 'gamma': 3.7262221658602357, 'reg_alpha': 0.055704055488356614, 'reg_lambda': 6.777471862379118e-05, 'early_stopping_rounds': 30}


[INFO 03-11 09:23:20] ax.service.ax_client: Completed trial 84 with data: {'avg_rmse_nonzero': (np.float64(4.641857), np.float64(0.702436))}.
[INFO 03-11 09:23:20] ax.service.ax_client: Saved JSON-serialized state of optimization to `c:\Users\Chris\OneDrive\Desktop\ML Project\ax_client_XGB_DF_fT_checkpoint.json`.
c:\Users\Chris\pytorch_gpu\Lib\site-packages\ax\core\data.py:295: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return cls(df=pd.concat(dfs, axis=0, sort=True))
c:\Users\Chris\pytorch_gpu\Lib\site-packages\ax\modelbridge\cross_validation.py:439: UserWarning: Encountered exception in computing model fit quality: RandomModelBridge does not support prediction.
  warn("Encountered exception in computing model fit quality: " + str(e))
[

...

Result: Non-zero RMSE = 4.1869

Trial 100/100


c:\Users\Chris\pytorch_gpu\Lib\site-packages\linear_operator\utils\cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
C:\Users\Chris\python\Lib\dataclasses.py:1581: RuntimeWarning: If using a 2-dim `batch_initial_conditions` botorch will default to old behavior of ignoring `num_restarts` and just use the given `batch_initial_conditions` by setting `raw_samples` to None.
  return obj.__class__(**changes)
c:\Users\Chris\pytorch_gpu\Lib\site-packages\linear_operator\utils\cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
C:\Users\Chris\python\Lib\dataclasses.py:1581: RuntimeWarning: If using a 2-dim `batch_initial_conditions` botorch will default to old behavior of ignoring `num_restarts` and just use the given `batch_initial_conditions` by setting `raw_samples` to None.
  return obj.__class__(**changes)
c:\Users\Chris\pytorch_gpu\Lib\site-packages\linear_operator\utils\cholesky.py:40:

Parameters: {'n_estimators': 638, 'max_depth': 3, 'learning_rate': 0.005, 'subsample': 1.0, 'colsample_bytree': 0.3, 'min_child_weight': 6, 'gamma': 3.162864367783304, 'reg_alpha': 10.0, 'reg_lambda': 0.01960647310709318, 'early_stopping_rounds': 27}


[INFO 03-16 10:32:10] ax.service.ax_client: Completed trial 99 with data: {'avg_rmse_nonzero': (np.float64(4.489736), np.float64(0.747362))}.
[INFO 03-16 10:32:10] ax.service.ax_client: Saved JSON-serialized state of optimization to `c:\Users\Chris\OneDrive\Desktop\ML Project\ax_client_XGB_DF_fT_checkpoint.json`.
c:\Users\Chris\pytorch_gpu\Lib\site-packages\ax\core\data.py:295: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return cls(df=pd.concat(dfs, axis=0, sort=True))


Result: Non-zero RMSE = 4.4897

OPTIMIZATION COMPLETE
Best parameters: {'n_estimators': 3742, 'max_depth': 11, 'learning_rate': 0.04133057413580116, 'subsample': 0.6830441900528967, 'colsample_bytree': 0.32020963728427887, 'min_child_weight': 8, 'gamma': 3.1530884513631463, 'reg_alpha': 0.16534358452677872, 'reg_lambda': 0.0001841925425023652, 'early_stopping_rounds': 16}
Best Non-zero RMSE: 4.3778


In [15]:
ax_client_xgb_df = AxClient.load_from_json_file(SAVE_PATH_XGB_DF)
# Get best parameters
best_parameters_xgb_df, values_xgb_df = ax_client_xgb_df.get_best_parameters()
print(f"\n{'='*60}")
print("OPTIMIZATION COMPLETE — XGBoost Phase Fractions DF f(T)")
print('='*60)
print(f"Best parameters: {best_parameters_xgb_df}")
print(f"Best Non-zero RMSE: {values_xgb_df[0]['avg_rmse_nonzero']:.4f}")

[INFO 03-09 09:01:32] ax.service.ax_client: Starting optimization with verbose logging. To disable logging, set the `verbose_logging` argument to `False`. Note that float values in the logs are rounded to 6 decimal points.



OPTIMIZATION COMPLETE — XGBoost Phase Fractions DF f(T)
Best parameters: {'n_estimators': 1203, 'max_depth': 18, 'learning_rate': 0.022103, 'subsample': 0.69082, 'colsample_bytree': 0.436275, 'min_child_weight': 1, 'gamma': 3.34254, 'reg_alpha': 0.004137, 'reg_lambda': 0.10817, 'early_stopping_rounds': 23}
Best Non-zero RMSE: 4.1923


c:\Users\Chris\pytorch_gpu\Lib\site-packages\ax\core\data.py:295: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return cls(df=pd.concat(dfs, axis=0, sort=True))
c:\Users\Chris\pytorch_gpu\Lib\site-packages\ax\core\data.py:295: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return cls(df=pd.concat(dfs, axis=0, sort=True))
